# Data Extraction Pipeline (Stages 1–4)
This notebook extracts workflow metadata, run-level metrics, step telemetry (TTFTS), and workload signatures (including artifact-based executed-test evidence).

## Stage 1 — Verify workflows and label execution-style evidence

In [ ]:
# ============================================================
# Stage 1 (UPGRADED): Verify GitHub Actions workflows from URL list (V16->Aligned)
#
# Goal:
#   - Make Stage 1 style + invocation_types detection match 100% with
#     the "1 - Method - 3.1.1 - Instru Test Signal V16" core method.
#   - Only exception: GMD is workflow-only (YAML/called-files), no build.gradle.
#
# Output labels:
#   styles: Emu_Community, Emu_Custom, Third-Party, Real-Device, GMD
#   invocation_types: Gradle_Connected, Gradle_GMD, Gradle_BaselineProfile,
#                     ADB_Am_Instrument, 3P_CLIs, Flutter_Driver, UNKNOWN
#
# Notes:
#   - Third-Party style is emitted as soon as 3P invocation is detected.
#   - Real-Device style is emitted when real-device ADB is detected.
#   - GMD is "capable" only based on YAML/called-files strong signals.
# ============================================================

import base64
import csv
import random
import re
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple, Union

import requests

try:
    from tqdm import tqdm
except ImportError:
    tqdm = None

# ============================================================
# CONFIG
# ============================================================
TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

URL_LIST_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext")
URL_LIST_BASENAME = "URL_List.csv"

OUT_DIR = URL_LIST_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_VERIFIED_WORKFLOWS_CSV = OUT_DIR / "verified_workflows_v16.csv"

DEFAULT_BRANCH_ONLY = True
FOLLOW_CALLED_FILES_GH = True
MAX_FOLLOW_DEPTH_GH = 2
MAX_FOLLOW_BYTES_GH = 1_500_000

CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 60
MAX_RETRIES_PER_REQUEST = 8
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60
MAX_PAGES_PER_LIST = 2000


# ============================================================
# Helpers
# ============================================================
def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def dot_fullname_to_slash(s: str) -> str:
    s = (s or "").strip()
    if not s:
        return ""
    s = s.replace("https://github.com/", "").replace("http://github.com/", "")
    s = s.split("?")[0].strip().strip("/")
    if "/" in s:
        return s
    if "." in s:
        owner, repo = s.split(".", 1)
        return f"{owner}/{repo}"
    return s

def b64_to_text(content_b64: str) -> Optional[str]:
    try:
        return base64.b64decode(content_b64).decode("utf-8", errors="ignore")
    except Exception:
        return None

def ensure_csv_header(csv_path: Path, fieldnames: List[str]) -> None:
    if csv_path.exists():
        return
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with csv_path.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()

def append_row(csv_path: Path, fieldnames: List[str], row: Dict) -> None:
    with csv_path.open("a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writerow(row)

def unique_preserve(seq: Iterable[str]) -> List[str]:
    seen: Set[str] = set()
    out: List[str] = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def load_existing_keys(csv_path: Path, key_field: str) -> Set[str]:
    keys: Set[str] = set()
    if not csv_path.exists():
        return keys
    with csv_path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        rdr = csv.DictReader(f)
        for row in rdr:
            k = (row.get(key_field) or "").strip()
            if k:
                keys.add(k)
    return keys

def load_tokens_from_env_file(env_path: Path, max_tokens: int = 3) -> List[str]:
    if not env_path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {env_path}")

    tokens: List[str] = []
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN_") and v:
            tokens.append(v)
            if len(tokens) >= max_tokens:
                break

    if not tokens:
        raise ValueError(f"No tokens found in {env_path}. Expected keys like GITHUB_TOKEN_1=...")
    return tokens

def resolve_input_csv(folder: Path, basename: str) -> Path:
    p = folder / basename
    if p.exists():
        return p
    for cand in (basename.lower(), basename.upper()):
        q = folder / cand
        if q.exists():
            return q
    target = basename.lower()
    for f in folder.rglob("*.csv"):
        if f.name.lower() == target:
            return f
    for f in folder.rglob("*.CSV"):
        if f.name.lower() == target:
            return f
    raise FileNotFoundError(f"Could not find '{basename}' under: {folder}")

def load_repo_fullnames_from_single_csv(csv_path: Path) -> List[str]:
    if not csv_path.exists():
        raise FileNotFoundError(f"Input CSV not found: {csv_path}")

    candidates = (
        "full_name", "repo", "repository", "url", "repo_url", "github_url",
        "repository_url", "repo link", "repo_link", "project",
    )

    repos: List[str] = []
    with csv_path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        rdr = csv.DictReader(f)
        if not rdr.fieldnames:
            raise RuntimeError(f"{csv_path.name} has no header row.")
        cols = [c.lower().strip() for c in rdr.fieldnames]
        header_map = {c.lower().strip(): c for c in rdr.fieldnames}

        col = None
        for cand in candidates:
            if cand in cols:
                col = cand
                break
        if not col:
            for c in cols:
                if "repo" in c and "url" in c:
                    col = c
                    break
                if c in ("url", "link", "github"):
                    col = c
                    break
        if not col:
            raise RuntimeError(
                f"No supported repo column in {csv_path.name}. "
                f"Found headers: {rdr.fieldnames}. Expected one of: {candidates}"
            )

        real_col = header_map[col]
        for row in rdr:
            v = (row.get(real_col) or "").strip()
            if not v:
                continue
            repo = dot_fullname_to_slash(v)
            if repo and "/" in repo:
                repos.append(repo)

    repos = unique_preserve(repos)
    print(f"Loaded {len(repos)} repos from: {csv_path.name} (column='{real_col}')")
    return repos


# ============================================================
# GitHub API client
# ============================================================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None

class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "workflow-verifier-v16-aligned/2.0",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        now = int(time.time())
        candidates = []
        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                candidates.append((0, i))
            else:
                if st.reset_epoch is not None and st.reset_epoch <= now:
                    candidates.append((0, i))
                else:
                    candidates.append((1, i))
        candidates.sort()
        return candidates[0][1]

    def _sleep_until_reset(self) -> None:
        now = int(time.time())
        resets = [st.reset_epoch for st in self.tokens if st.reset_epoch]
        if not resets:
            time.sleep(5)
            return
        soonest = min(resets)
        sleep_s = max(1, soonest - now + 2)
        print(f"[rate-limit] sleeping {sleep_s}s until reset...")
        time.sleep(sleep_s)

    def _backoff(self, attempt: int) -> None:
        sleep_s = min(BACKOFF_CAP_S, (BACKOFF_BASE_S ** attempt)) + random.random()
        time.sleep(sleep_s)

    def request_json(self, method: str, url: str, params: Optional[Dict] = None) -> Optional[Union[Dict, List]]:
        last_status = None
        for attempt in range(1, MAX_RETRIES_PER_REQUEST + 1):
            idx = self._pick_idx()
            st = self.tokens[idx]
            self.session.headers["Authorization"] = f"Bearer {st.token}"

            try:
                resp = self.session.request(method, url, params=params, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            except requests.exceptions.RequestException as e:
                print(f"[net] {method} {url} attempt {attempt}: {e}")
                self._backoff(attempt)
                continue

            last_status = resp.status_code

            rem = resp.headers.get("X-RateLimit-Remaining")
            if rem is not None:
                try: st.remaining = int(rem)
                except Exception: pass
            rst = resp.headers.get("X-RateLimit-Reset")
            if rst is not None:
                try: st.reset_epoch = int(rst)
                except Exception: pass

            if resp.status_code == 404:
                return None

            retry_after = resp.headers.get("Retry-After")
            if resp.status_code in (403, 429) and retry_after:
                try:
                    ra = int(retry_after)
                    time.sleep(min(BACKOFF_CAP_S, max(1, ra)) + random.random())
                    continue
                except Exception:
                    pass

            text_l = (resp.text or "").lower()
            if resp.status_code in (403, 429) and (
                "rate limit" in text_l
                or "secondary rate limit" in text_l
                or "abuse detection" in text_l
                or "too many requests" in text_l
            ):
                if any((t.remaining is None) or (t.remaining > 0) for t in self.tokens):
                    self._backoff(attempt)
                    continue
                self._sleep_until_reset()
                continue

            if resp.status_code in (500, 502, 503, 504):
                self._backoff(attempt)
                continue

            if resp.status_code >= 400:
                return None

            try:
                return resp.json()
            except Exception:
                return None

        print(f"[giveup] {method} {url} after {MAX_RETRIES_PER_REQUEST} tries (last_status={last_status})")
        return None

    def paginate(self, url: str, params: Optional[Dict], item_key: str) -> Iterable[Dict]:
        page = 1
        while page <= MAX_PAGES_PER_LIST:
            p = dict(params or {})
            p.update({"per_page": 100, "page": page})
            data = self.request_json("GET", url, params=p)
            if data is None:
                return
            items = data if isinstance(data, list) else data.get(item_key, [])
            if not items:
                return
            for it in items:
                yield it
            if isinstance(items, list) and len(items) < 100:
                return
            page += 1


# ============================================================
# GitHub endpoints
# ============================================================
def get_repo_default_branch(gh: GitHubClient, full_name: str) -> str:
    data = gh.request_json("GET", f"https://api.github.com/repos/{full_name}")
    if not data or not isinstance(data, dict):
        return ""
    return (data.get("default_branch") or "").strip()

def list_workflows(gh: GitHubClient, full_name: str) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/workflows"
    return list(gh.paginate(url, params={}, item_key="workflows"))

def get_workflow_meta(gh: GitHubClient, full_name: str, workflow_identifier: str) -> Optional[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/workflows/{workflow_identifier}"
    return gh.request_json("GET", url)

def fetch_repo_file_text(gh: GitHubClient, full_name: str, repo_path: str, ref: Optional[str]) -> Tuple[str, Optional[int]]:
    url = f"https://api.github.com/repos/{full_name}/contents/{repo_path.lstrip('/')}"
    params = {"ref": ref} if ref else None
    data = gh.request_json("GET", url, params=params)
    if not data or not isinstance(data, dict):
        return "", None

    size = None
    try:
        if data.get("size") is not None:
            size = int(data.get("size"))
    except Exception:
        size = None

    if data.get("encoding") == "base64" and data.get("content"):
        return (b64_to_text(data["content"]) or ""), size

    dl = data.get("download_url")
    if dl:
        try:
            r = gh.session.get(dl, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            if r.status_code == 200:
                return (r.text or ""), size
        except requests.exceptions.RequestException:
            return "", size

    return "", size


# ============================================================
# Core-method-aligned preprocessing + signal extraction
# (mirrors your "Instru Test Signal V16" notebook logic)
# ============================================================
def compile_any(patterns: List[str], flags=0) -> re.Pattern:
    return re.compile("|".join(f"(?:{p})" for p in patterns), flags)

COMMENT_LINE_RE = re.compile(r'(?m)^\s*(#|//|REM\b|::).*?$')

def strip_comments(raw: str) -> str:
    return COMMENT_LINE_RE.sub("", raw or "")

def normalize_block_keys(text: str) -> str:
    text = re.sub(r'(?mi)^\s*(script|run|command)\s*:\s*(?!\||>|\|\-)\s*(.+)$', r'\2', text)
    text = re.sub(r'(?mi)^\s*(script|run|command)\s*:\s*(\||>|\|\-)\s*$', '', text)
    text = re.sub(
        r'(?m)^(\s*)-\s*(?=(?:\./|\.\\|bash|sh|pwsh|powershell|gradle(?:w)?|adb|flutter|gcloud|saucectl|appcenter)\b)',
        r'\1', text
    )
    return text

GHA_EXPR_RE = re.compile(r"\${{\s*[^}]+}}")

EXCLUDED_TASK_SEGMENT_RE = re.compile(
    r'(^|\s)(?:-x|--exclude-task)\s+(["\']?)[:\w\.-]*(?:androidtest|baselineprofile)[\w:\.-]*\2\b',
    re.IGNORECASE | re.MULTILINE,
)

def pre_sanitize(text: str) -> str:
    t = EXCLUDED_TASK_SEGMENT_RE.sub(lambda m: (m.group(1) or " "), text or "")
    return GHA_EXPR_RE.sub("", t or "")

def _prepare_content(text: str) -> str:
    content = strip_comments(text or "")
    content = normalize_block_keys(content)
    content = pre_sanitize(content)
    return content

# --- device/env patterns (core aligned) ---
DEVICE_PATTERNS = [
    # community emulator actions
    r"(?mi)^\s*uses\s*:\s*(reactivecircus/android-emulator-runner|malinskiy/action-android/emulator-run-cmd|hannesa2/action-android/emulator-run-cmd|vgaidarji/android-github-actions-emulator)@",
    # other emulator actions (android + emulator/avd in name)
    r"(?mi)^\s*uses\s*:\s*(?!reactivecircus/android-emulator-runner@)(?!malinskiy/action-android/emulator-run-cmd@)(?!hannesa2/action-android/emulator-run-cmd@)(?!emulator-wtf/run-tests@)[\w\.-]+/[\w\./-]*android[\w\./-]*(?:\bemulator\b|\bavd\b)[\w\./-]*@",
    # custom emulator runtime setup
    r"(?mi)\b(emulator\b[^\n]*-avd\s+\S+|adb\s+wait[- ]?for[- ]?device\b|adb\s+-s\s+emulator-\d+\b|\bandroid-wait-for-emulator\b|\bstart-emulator\.sh\b|\bavdmanager\b|\bandroid\b[^\n]*\bcreate\s+avd\b)\b",
    # real device ADB (exclude emulator/local)
    r"(?mi)\badb\s+-s\s+(?!emulator-\d+\b)(?!localhost:\d+\b)(?!127\.0\.0\.1:\d+\b)\S+\b",
    # third-party env/actions
    r"(?mi)\b(gcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run|\bappcenter\s+test\s+run\s+android\b|\bsaucectl\b|\b(browserstack|bstack)\b|\bmaestro\s+cloud\b|\bemulator\.wtf\b|^\s*uses\s*:\s*emulator-wtf/run-tests@)\b",
]
DEVICE_RE = compile_any(DEVICE_PATTERNS, flags=re.IGNORECASE)

# --- invocation triggers (core aligned) ---
TRIGGER_PATTERNS_PRIMARY = {
    "gradle_tasks_connected": [
        r"(?mi)\bconnected\w*androidtest\b",   # includes connectedDebugAndroidTest etc
        r"(?mi)\bconnectedcheck\b",
        r"(?mi)\bdevicecheck\b",
        r"(?mi)\balldevicescheck\b",
    ],
    "gradle_tasks_baselineprofile": [
        r"(?mi)\bgenerate\w*baselineprofile\b",
        r"(?mi)\bcollect\w*baselineprofile\b",
    ],
    "adb_am_instrument": [
        r"(?mi)\bam\s+instrument\b",
    ],
    "third_party_cli": [
        r"(?mi)\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run\b",
        r"(?mi)\bflank\s+android\s+run\b",
        r"(?mi)\bappcenter\s+test\s+run\s+android\b",
        r"(?mi)\bsaucectl\b",
        r"(?mi)\b(browserstack|bstack)\b",
        r"(?mi)\bmaestro\s+cloud\b",
        r"(?mi)\bemulator\.wtf\b",
        r"(?mi)^\s*uses\s*:\s*emulator-wtf/run-tests@",
    ],
    "flutter_driver": [
        r"(?mi)\bflutter\s+drive\b",
        r"(?mi)\bflutter\s+test\s+integration_test\b",
    ],
}

# --- GMD (workflow-only strong signals, core aligned) ---
RE_MANAGEDDEVICES_PROP = re.compile(
    r"-Pandroid\.(?:testoptions\.manageddevices|experimental\.testOptions\.managedDevices)\.[\w\.\-]+",
    re.IGNORECASE
)
RE_GMD_TASK = re.compile(
    r"(?i)\b(:?[\w\.-]+:)?(managedDeviceCheck|managedDevice\w*AndroidTest|allDevicesCheck|allDevices\w*AndroidTest)\b"
)
RE_GRADLE_CMD = re.compile(r"(\./gradlew\b|gradlew\.bat\b|gradle\s+)", re.IGNORECASE)
RE_EXCLUDE_CONNECTED = re.compile(r"\bconnected[\w:\-]*androidtest\b", re.IGNORECASE)
RE_EXCLUDE_SPOON_MARATHON = re.compile(r"\b(spoon|marathon)\b", re.IGNORECASE)

def has_gmd_gradle_trigger(content: str, third_party_in_block: bool) -> bool:
    if third_party_in_block:
        return False
    if RE_MANAGEDDEVICES_PROP.search(content or ""):
        return True
    for line in (content or "").splitlines():
        if not RE_GRADLE_CMD.search(line):
            continue
        if RE_EXCLUDE_SPOON_MARATHON.search(line):
            continue
        if RE_EXCLUDE_CONNECTED.search(line):
            continue
        if RE_GMD_TASK.search(line):
            return True
    return False

def has_connected_gradle_trigger(content: str) -> bool:
    # connected*AndroidTest etc (variant-safe)
    return bool(re.search(r"(?mi)\bconnected\w*androidtest\b|\bconnectedcheck\b|\bdevicecheck\b|\balldevicescheck\b", content or ""))

def has_baselineprofile_trigger(content: str) -> bool:
    return bool(re.search(r"(?mi)\bgenerate\w*baselineprofile\b|\bcollect\w*baselineprofile\b", content or ""))

def scan_text_for_signals(text: str) -> Tuple[Set[str], Dict[str, bool], Dict[str, bool], bool]:
    """
    Returns:
      device_labels: set[str] (raw device/env matches)
      groups: dict of core booleans for exec-env mapping
      triggers: dict of invocation booleans
      flutter_flag: bool
    """
    content = _prepare_content(text or "")
    low = (content or "").lower()

    # triggers
    triggers: Dict[str, bool] = {}
    for k, pats in TRIGGER_PATTERNS_PRIMARY.items():
        triggers[k] = any(re.search(p, content or "") for p in pats)

    third_party_in_block = bool(triggers.get("third_party_cli"))

    # groups (exec-env mapping)
    groups: Dict[str, bool] = {
        "emu_action_comm": bool(re.search(r"(?mi)^\s*uses\s*:\s*(reactivecircus/android-emulator-runner|malinskiy/action-android/emulator-run-cmd|hannesa2/action-android/emulator-run-cmd|vgaidarji/android-github-actions-emulator)@", content or "")),
        "emu_action_other": bool(re.search(r"(?mi)^\s*uses\s*:\s*(?!reactivecircus/android-emulator-runner@)(?!malinskiy/action-android/emulator-run-cmd@)(?!hannesa2/action-android/emulator-run-cmd@)(?!emulator-wtf/run-tests@)[\w\.-]+/[\w\./-]*android[\w\./-]*(?:\bemulator\b|\bavd\b)[\w\./-]*@", content or "")),
        "emu_custom_runtime": bool(re.search(r"(?mi)\b(emulator\b[^\n]*-avd\s+\S+|adb\s+wait[- ]?for[- ]?device\b|adb\s+-s\s+emulator-\d+\b|\bandroid-wait-for-emulator\b|\bstart-emulator\.sh\b|\bavdmanager\b|\bandroid\b[^\n]*\bcreate\s+avd\b)\b", content or "")),
        "real_device_adb": bool(re.search(r"(?mi)\badb\s+-s\s+(?!emulator-\d+\b)(?!localhost:\d+\b)(?!127\.0\.0\.1:\d+\b)\S+\b", content or "")),
        "third_party_env": bool(re.search(r"(?mi)\b(gcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run|\bappcenter\s+test\s+run\s+android\b|\bsaucectl\b|\b(browserstack|bstack)\b|\bmaestro\s+cloud\b|\bemulator\.wtf\b|^\s*uses\s*:\s*emulator-wtf/run-tests@)\b", content or "")),
        "gmd_capable": has_gmd_gradle_trigger(content, third_party_in_block=third_party_in_block),
        "connected_gradle": has_connected_gradle_trigger(content),
        "baselineprofile_gradle": has_baselineprofile_trigger(content),
    }

    # device labels (optional evidence labels)
    device_labels: Set[str] = set()
    if groups["emu_custom_runtime"]:
        device_labels.add("emu_custom_runtime")
    if groups["emu_action_comm"] or groups["emu_action_other"]:
        device_labels.add("emu_community_action")
    if groups["real_device_adb"]:
        device_labels.add("real_device_adb")
    if groups["third_party_env"]:
        device_labels.add("third_party_env")
    if groups["gmd_capable"]:
        device_labels.add("gradle_gmd_capable")
    if groups["connected_gradle"]:
        device_labels.add("gradle_connected")
    if groups["baselineprofile_gradle"]:
        device_labels.add("gradle_baselineprofile")
    if triggers.get("adb_am_instrument"):
        device_labels.add("adb_am_instrument")
    if triggers.get("third_party_cli"):
        device_labels.add("third_party_invoke")
    if triggers.get("flutter_driver"):
        device_labels.add("flutter_driver")

    flutter_flag = bool(triggers.get("flutter_driver"))

    return device_labels, groups, triggers, flutter_flag

def map_exec_env_style(groups: Dict[str, bool]) -> Set[str]:
    """
    Core-method mapping + your label normalization:
      - Third-Party and Real-Device are emitted using hyphenated labels.
      - GMD is "capable" label (workflow-only).
    """
    styles: Set[str] = set()

    # Emu custom
    if groups.get("emu_custom_runtime"):
        styles.add("Emu_Custom")

    # Emu community (known or other emulator actions)
    if groups.get("emu_action_comm") or groups.get("emu_action_other"):
        styles.add("Emu_Community")

    # Real device
    if groups.get("real_device_adb"):
        styles.add("Real-Device")

    # Third party environment (BUT your plan says: style can be set as soon as 3P invocation is detected;
    # we enforce that upstream, see classify_workflow below)
    if groups.get("third_party_env"):
        styles.add("Third-Party")

    # GMD (capable from YAML)
    if groups.get("gmd_capable"):
        styles.add("GMD")

    return styles

def map_test_invocations(groups: Dict[str, bool], triggers: Dict[str, bool], flutter_flag: bool) -> Set[str]:
    inv: Set[str] = set()

    if groups.get("connected_gradle") or triggers.get("gradle_tasks_connected"):
        inv.add("Gradle_Connected")
    if groups.get("gmd_capable"):
        inv.add("Gradle_GMD")
    if groups.get("baselineprofile_gradle") or triggers.get("gradle_tasks_baselineprofile"):
        inv.add("Gradle_BaselineProfile")
    if triggers.get("adb_am_instrument"):
        inv.add("ADB_Am_Instrument")
    if triggers.get("third_party_cli"):
        inv.add("3P_CLIs")
    if flutter_flag:
        inv.add("Flutter_Driver")

    if not inv:
        inv.add("UNKNOWN")

    return inv


# ============================================================
# Called-file following (kept from your Stage 1)
# ============================================================
LOCAL_USES_RE = re.compile(r'(?mi)^\s*uses\s*:\s*(?P<ref>\./\S+?)(?:\s+#.*)?$')
WORKDIR_RE = re.compile(r'(?mi)^\s*working-directory\s*:\s*(?P<wd>[^\n#]+)')
SCRIPT_CALL_RE = re.compile(r'''(?mix)
(?:^|[;&|()\s"'`])
(?:(?:bash|sh|pwsh|powershell|python|python3|node|ruby)\s+)?(?P<path>(?:\./|\.\\)?[\w./\\-]+\.(?:sh|ps1|bat|cmd|py|js|rb|pl))
(?:\s|$)
''')
CONFIG_ARG_RE = re.compile(r'(?mi)\b--config(?:=|\s+)(?P<path>[^\s"\']+)')
NO_FOLLOW_BASENAMES = {"gradlew", "gradlew.bat", "gradle", "adb", "flutter"}

def _strip_quotes(s: str) -> str:
    return (s or "").strip().strip('"').strip("'").strip("`")

def is_dynamic_ref(ref: str) -> bool:
    r = ref or ""
    return ("${{" in r) or ("${" in r) or ("$(" in r) or ("%{" in r)

def extract_workdirs(text: str) -> List[str]:
    out = []
    for m in WORKDIR_RE.finditer(text or ""):
        wd = _strip_quotes(m.group("wd"))
        if wd:
            wd = wd.replace("\\", "/").lstrip("./").strip("/")
            out.append(wd)
    return unique_preserve(out)

def extract_references(text: str) -> List[str]:
    refs: List[str] = []
    for m in LOCAL_USES_RE.finditer(text or ""):
        ref = _strip_quotes(m.group("ref"))
        if "@" in ref:
            ref = ref.split("@", 1)[0]
        refs.append(ref)
    for m in SCRIPT_CALL_RE.finditer(text or ""):
        refs.append(_strip_quotes(m.group("path")))
    for m in CONFIG_ARG_RE.finditer(text or ""):
        refs.append(_strip_quotes(m.group("path")))

    out = []
    for r in refs:
        if not r:
            continue
        base = Path(r.replace("\\", "/")).name.lower()
        if base in NO_FOLLOW_BASENAMES:
            continue
        out.append(r.replace("\\", "/").strip())
    return unique_preserve(out)

def resolve_repo_paths(ref: str, workdirs: List[str]) -> List[str]:
    if not ref or is_dynamic_ref(ref):
        return []
    r = _strip_quotes(ref.replace("\\", "/").strip())
    if r.startswith("./"):
        r = r[2:]
    r = r.lstrip("/")

    prefixes = [""] + [wd.strip("/") for wd in (workdirs or []) if wd]
    candidates: List[str] = []
    for pref in prefixes:
        p = f"{pref}/{r}".strip("/") if pref else r
        candidates.append(p.strip("/"))

    out: List[str] = []
    for p in candidates:
        suffix = Path(p).suffix.lower()
        if not suffix:
            out.append(f"{p}/action.yml")
            out.append(f"{p}/action.yaml")
        out.append(p)
    return unique_preserve([x for x in out if x])

_GH_TEXT_CACHE: Dict[Tuple[str, str, str], str] = {}
def gh_fetch_text_cached(gh: GitHubClient, full_name: str, path: str, ref: Optional[str]) -> str:
    key = (full_name, path, ref or "")
    if key in _GH_TEXT_CACHE:
        return _GH_TEXT_CACHE[key]
    text, size = fetch_repo_file_text(gh, full_name, path, ref)
    if size is not None and size > MAX_FOLLOW_BYTES_GH:
        text = ""
    _GH_TEXT_CACHE[key] = text or ""
    return _GH_TEXT_CACHE[key]


# ============================================================
# Job splitter (kept)
# ============================================================
JOBS_ANCHOR_RE = re.compile(r'(?m)^(?P<indent>\s*)jobs\s*:\s*$')
ANY_KEY_RE     = re.compile(r'(?m)^(?P<indent>\s*)(?P<name>[\w-]+)\s*:\s*$')

def split_jobs_blocks(raw: str) -> List[Tuple[str, str]]:
    m = JOBS_ANCHOR_RE.search(raw or "")
    if not m:
        return [("__whole__", raw or "")]
    jobs_indent = len(m.group("indent"))
    lines = (raw or "").splitlines(True)
    start_idx = (raw or "")[:m.end()].count("\n")
    candidates = []
    for i in range(start_idx, len(lines)):
        lm = ANY_KEY_RE.match(lines[i])
        if not lm:
            continue
        indent = len(lm.group("indent"))
        if indent > jobs_indent:
            candidates.append((i, indent, lm.group("name")))
    if not candidates:
        return [("__whole__", raw or "")]
    min_indent = min(indent for _, indent, _ in candidates)
    job_headers = [(i, name) for (i, indent, name) in candidates if indent == min_indent]
    if not job_headers:
        return [("__whole__", raw or "")]
    blocks = []
    header_indices = [i for i, _ in job_headers] + [len(lines)]
    for idx in range(len(job_headers)):
        i, name = job_headers[idx]
        j = header_indices[idx + 1]
        blocks.append((name, "".join(lines[i:j])))
    return blocks


# ============================================================
# Workflow classifier (aligned)
# ============================================================
def classify_workflow_aligned(
    gh: GitHubClient,
    full_name: str,
    wf_yaml_text: str,
    ref: Optional[str],
    follow_called_files: bool = True,
    max_depth: int = 2,
) -> Tuple[Set[str], Set[str], Set[str], bool, int, int]:
    """
    Returns:
      styles, invocations, evidence_labels, gmd_capable, followed_cnt, unresolved_cnt
    """
    agg_styles: Set[str] = set()
    agg_inv: Set[str] = set()
    agg_evidence: Set[str] = set()

    gmd_capable_any = False
    followed_files_count = 0
    unresolved_dynamic_refs_count = 0

    # 1) scan each job block (important for "3P block guards")
    for _, block in split_jobs_blocks(wf_yaml_text or ""):
        ev, groups, triggers, flutter_flag = scan_text_for_signals(block)
        agg_evidence |= ev

        # invocations
        inv = map_test_invocations(groups, triggers, flutter_flag)
        agg_inv |= inv

        # styles:
        styles = map_exec_env_style(groups)

        # Your plan: if 3P invocation is detected, style becomes Third-Party even if no explicit env setup
        if "3P_CLIs" in inv:
            styles.add("Third-Party")

        agg_styles |= styles
        if "GMD" in styles:
            gmd_capable_any = True

    # 2) follow called files (merge signals)
    if follow_called_files and wf_yaml_text:
        visited: Set[str] = set()
        queue: List[Tuple[str, int, List[str]]] = []

        refs_from_yaml = extract_references(wf_yaml_text)
        workdirs_from_yaml = extract_workdirs(wf_yaml_text)

        for r in refs_from_yaml:
            if is_dynamic_ref(r):
                unresolved_dynamic_refs_count += 1
                continue
            for p in resolve_repo_paths(r, workdirs_from_yaml):
                queue.append((p, 0, workdirs_from_yaml))

        while queue:
            path, depth, wds = queue.pop(0)
            if depth > max_depth or not path or path in visited:
                continue
            visited.add(path)

            txt = gh_fetch_text_cached(gh, full_name, path, ref=ref)
            if not txt:
                continue
            followed_files_count += 1

            ev, groups, triggers, flutter_flag = scan_text_for_signals(txt)
            agg_evidence |= ev

            inv = map_test_invocations(groups, triggers, flutter_flag)
            agg_inv |= inv

            styles = map_exec_env_style(groups)
            if "3P_CLIs" in inv:
                styles.add("Third-Party")

            agg_styles |= styles
            if "GMD" in styles:
                gmd_capable_any = True

            if depth == max_depth:
                continue

            child_refs = extract_references(txt)
            child_wds = extract_workdirs(txt)

            for rr in child_refs:
                if is_dynamic_ref(rr):
                    unresolved_dynamic_refs_count += 1
                    continue
                for pp in resolve_repo_paths(rr, child_wds):
                    queue.append((pp, depth + 1, child_wds))

    # Ensure invocation_types never empty
    if not agg_inv:
        agg_inv.add("UNKNOWN")

    return agg_styles, agg_inv, agg_evidence, gmd_capable_any, followed_files_count, unresolved_dynamic_refs_count


# ============================================================
# MAIN
# ============================================================
def main() -> None:
    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH, max_tokens=3)
    gh = GitHubClient(tokens)

    url_list_file = resolve_input_csv(URL_LIST_DIR, URL_LIST_BASENAME)
    print("Using input:", url_list_file)

    repos = load_repo_fullnames_from_single_csv(url_list_file)
    if not repos:
        raise RuntimeError(f"No repos found in input CSV: {url_list_file}")

    out_fields = [
        "full_name",
        "default_branch",
        "has_any_gha_workflow",
        "workflow_identifier",
        "workflow_id",
        "workflow_name",
        "workflow_path",
        "workflow_html_url",

        "looks_like_instru",
        "gmd_capable",
        "styles",
        "invocation_types",
        "evidence_labels",
        "followed_files_count",
        "unresolved_dynamic_refs_count",
        "scanned_at_utc",
        "label_source",
    ]
    ensure_csv_header(OUT_VERIFIED_WORKFLOWS_CSV, out_fields)
    existing = load_existing_keys(OUT_VERIFIED_WORKFLOWS_CSV, "workflow_id")

    repo_iter = repos
    if tqdm is not None:
        repo_iter = tqdm(repos, desc="Stage1: verify workflows + detect styles (aligned)")

    default_branch_cache: Dict[str, str] = {}

    for full_name in repo_iter:
        if DEFAULT_BRANCH_ONLY:
            if full_name not in default_branch_cache:
                default_branch_cache[full_name] = get_repo_default_branch(gh, full_name)
            default_branch = default_branch_cache[full_name]
            if not default_branch:
                continue
        else:
            default_branch = ""

        ref = default_branch if DEFAULT_BRANCH_ONLY else None

        workflows = list_workflows(gh, full_name) or []
        has_any = "yes" if workflows else "no"
        if not workflows:
            continue

        for wf in workflows:
            wf_id = str(wf.get("id") or "").strip()
            if wf_id and wf_id in existing:
                continue

            wf_name = (wf.get("name") or "").strip()
            wf_path = (wf.get("path") or "").strip()
            wf_html = (wf.get("html_url") or "").strip()
            wf_identifier = (wf_path.split("/")[-1] if wf_path else str(wf.get("path") or ""))

            meta = get_workflow_meta(gh, full_name, wf_identifier) or {}
            if isinstance(meta, dict):
                wf_name = (meta.get("name") or wf_name).strip()
                wf_path = (meta.get("path") or wf_path).strip()
                wf_html = (meta.get("html_url") or wf_html).strip()
                wf_id = str(meta.get("id") or wf_id).strip()

            if not wf_path:
                continue

            yml_text, size = fetch_repo_file_text(gh, full_name, wf_path, ref=ref)
            if size is not None and size > MAX_FOLLOW_BYTES_GH:
                yml_text = ""

            styles, inv, evidence, gmd_capable, followed_cnt, unresolved_cnt = classify_workflow_aligned(
                gh=gh,
                full_name=full_name,
                wf_yaml_text=yml_text or "",
                ref=ref,
                follow_called_files=FOLLOW_CALLED_FILES_GH,
                max_depth=MAX_FOLLOW_DEPTH_GH,
            )

            # looks_like_instru: same meaning as before (any style or any invocation)
            looks_like_instru = bool(styles or (inv and inv != {"UNKNOWN"}))

            append_row(OUT_VERIFIED_WORKFLOWS_CSV, out_fields, {
                "full_name": full_name,
                "default_branch": default_branch,
                "has_any_gha_workflow": has_any,
                "workflow_identifier": wf_identifier,
                "workflow_id": wf_id,
                "workflow_name": wf_name,
                "workflow_path": wf_path,
                "workflow_html_url": wf_html,

                "looks_like_instru": "yes" if looks_like_instru else "no",
                "gmd_capable": "yes" if gmd_capable else "no",
                "styles": ",".join(sorted(styles)),
                "invocation_types": ",".join(sorted(inv)),
                "evidence_labels": ",".join(sorted(evidence)),
                "followed_files_count": str(followed_cnt),
                "unresolved_dynamic_refs_count": str(unresolved_cnt),
                "scanned_at_utc": now_utc_iso(),
                "label_source": "stage1_yaml_scan_v16" if FOLLOW_CALLED_FILES_GH else "stage1_yaml_only_v16",

            })

            if wf_id:
                existing.add(wf_id)

    print("Done.")
    print("Verified workflows:", OUT_VERIFIED_WORKFLOWS_CSV)

if __name__ == "__main__":
    main()


Using input: C:\Android Mobile App\ICST2026_Ext\URL_List.csv
Loaded 482 repos from: URL_List.csv (column='repo_url')


Stage1: verify workflows + detect styles: 100%|██████████| 482/482 [14:22<00:00,  1.79s/it]  

Done.
Verified workflows: C:\Android Mobile App\ICST2026_Ext\verified_workflows_v16.csv


## Stage 2 — Extract run-level metrics and attach style labels

In [ ]:
# ============================================================
# Stage 2 (UPGRADED): Run inventory + run-level fallback metrics (S2_)
#
# Goal:
# - Keep Stage 2 as the run-inventory feeder for Stage 3
# - Add *explicit* S2_ fallback fields (derived from the SAME jobs endpoint)
#   so Stage 3 can use them as first-priority fallback when step telemetry
#   is missing or cannot be computed.
#
# Key upgrade:
# - Compute run timing from jobs window:
#     S2_run_started_at_jobs_min = min(job.started_at)
#     S2_run_ended_at_jobs_max   = max(job.completed_at)
#     S2_run_duration_seconds_jobs_window
#
# - Also provide S2_ instrumentation window heuristics (already computed in Stage2)
#   but stored under S2_ names to avoid confusion in Stage 3 outputs.
#
# Notes:
# - We KEEP your existing (non-prefixed) columns for backward compatibility.
# - Stage 3 should prefer its own fresh computation, then fallback to S2_.
# ============================================================

import csv
import random
import re
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Union, Tuple

import requests

try:
    from tqdm import tqdm
except ImportError:
    tqdm = None


# =========================
# CONFIG
# =========================
TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

ROOT_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext")

IN_VERIFIED_WORKFLOWS_CSV = ROOT_DIR / "verified_workflows_v16.csv"
OUT_RUN_INVENTORY_CSV = ROOT_DIR / "run_inventory.csv"

DEFAULT_BRANCH_ONLY = True
PROCESS_ONLY_LOOKS_LIKE_INSTRU = True
FETCH_JOBS_FOR_EACH_RUN = True

MAX_RUNS_PER_WORKFLOW: Optional[int] = None
RUN_CREATED_AT_AFTER: Optional[str] = None

CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 60
MAX_RETRIES_PER_REQUEST = 8
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60
MAX_PAGES_PER_LIST = 2000

MAX_TOKENS_TO_USE = 7
SLEEP_BETWEEN_WORKFLOWS_SEC = 0.05


# =========================
# Helpers
# =========================
def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def iso_to_dt(iso: Optional[str]) -> Optional[datetime]:
    if not iso:
        return None
    try:
        return datetime.fromisoformat(iso.replace("Z", "+00:00"))
    except Exception:
        return None

def dt_to_seconds(a: Optional[datetime], b: Optional[datetime]) -> Optional[int]:
    if not a or not b:
        return None
    try:
        sec = int((b - a).total_seconds())
        return sec if sec >= 0 else None
    except Exception:
        return None

def ensure_csv_header(csv_path: Path, fieldnames: List[str]) -> None:
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with csv_path.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()

def append_row(csv_path: Path, fieldnames: List[str], row: Dict) -> None:
    with csv_path.open("a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writerow(row)

def load_existing_keys(csv_path: Path, key_field: str) -> Set[str]:
    keys: Set[str] = set()
    if not csv_path.exists():
        return keys
    with csv_path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        rdr = csv.DictReader(f)
        for row in rdr:
            k = (row.get(key_field) or "").strip()
            if k:
                keys.add(k)
    return keys

def load_tokens_from_env_file(env_path: Path, max_tokens: int = 3) -> List[str]:
    if not env_path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {env_path}")

    tokens: List[str] = []
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN_") and v:
            tokens.append(v)
            if len(tokens) >= max_tokens:
                break

    if not tokens:
        raise ValueError(f"No tokens found in {env_path}. Expected keys like GITHUB_TOKEN_1=...")
    return tokens

def unique_preserve(seq: Iterable[str]) -> List[str]:
    seen: Set[str] = set()
    out: List[str] = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def safe_join_names(names: List[str], max_len: int = 500) -> str:
    s = ",".join(unique_preserve([n.strip() for n in names if n and n.strip()]))
    if len(s) <= max_len:
        return s
    return s[: max_len - 3] + "..."


# =========================
# GitHub API client
# =========================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None

class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "run-inventory-stage2-v16/1.3",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        now = int(time.time())
        candidates = []
        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                candidates.append((0, i))
            else:
                if st.reset_epoch is not None and st.reset_epoch <= now:
                    candidates.append((0, i))
                else:
                    candidates.append((1, i))
        candidates.sort()
        return candidates[0][1]

    def _sleep_until_reset(self) -> None:
        now = int(time.time())
        resets = [st.reset_epoch for st in self.tokens if st.reset_epoch]
        if not resets:
            time.sleep(5)
            return
        soonest = min(resets)
        sleep_s = max(1, soonest - now + 2)
        print(f"[rate-limit] sleeping {sleep_s}s until reset...")
        time.sleep(sleep_s)

    def _backoff(self, attempt: int) -> None:
        sleep_s = min(BACKOFF_CAP_S, (BACKOFF_BASE_S ** attempt)) + random.random()
        time.sleep(sleep_s)

    def request_json(self, method: str, url: str, params: Optional[Dict] = None) -> Optional[Union[Dict, List]]:
        last_status = None
        for attempt in range(1, MAX_RETRIES_PER_REQUEST + 1):
            idx = self._pick_idx()
            st = self.tokens[idx]
            self.session.headers["Authorization"] = f"Bearer {st.token}"

            try:
                resp = self.session.request(method, url, params=params, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            except requests.exceptions.RequestException:
                self._backoff(attempt)
                continue

            last_status = resp.status_code

            rem = resp.headers.get("X-RateLimit-Remaining")
            if rem is not None:
                try:
                    st.remaining = int(rem)
                except Exception:
                    pass
            rst = resp.headers.get("X-RateLimit-Reset")
            if rst is not None:
                try:
                    st.reset_epoch = int(rst)
                except Exception:
                    pass

            if resp.status_code == 404:
                return None

            retry_after = resp.headers.get("Retry-After")
            if resp.status_code in (403, 429) and retry_after:
                try:
                    ra = int(retry_after)
                    time.sleep(min(BACKOFF_CAP_S, max(1, ra)) + random.random())
                    continue
                except Exception:
                    pass

            text_l = (resp.text or "").lower()
            if resp.status_code in (403, 429) and (
                "rate limit" in text_l
                or "secondary rate limit" in text_l
                or "abuse detection" in text_l
                or "too many requests" in text_l
            ):
                if any((t.remaining is None) or (t.remaining > 0) for t in self.tokens):
                    self._backoff(attempt)
                    continue
                self._sleep_until_reset()
                continue

            if resp.status_code in (500, 502, 503, 504):
                self._backoff(attempt)
                continue

            if resp.status_code >= 400:
                return None

            try:
                return resp.json()
            except Exception:
                return None

        print(f"[giveup] {method} {url} after {MAX_RETRIES_PER_REQUEST} tries (last_status={last_status})")
        return None

    def paginate(self, url: str, params: Optional[Dict], item_key: str) -> Iterable[Dict]:
        page = 1
        while page <= MAX_PAGES_PER_LIST:
            p = dict(params or {})
            p.update({"per_page": 100, "page": page})
            data = self.request_json("GET", url, params=p)
            if data is None:
                return
            items = data if isinstance(data, list) else data.get(item_key, [])
            if not items:
                return
            for it in items:
                yield it
            if isinstance(items, list) and len(items) < 100:
                return
            page += 1


# =========================
# GitHub endpoints
# =========================
def get_repo_default_branch(gh: GitHubClient, full_name: str) -> str:
    data = gh.request_json("GET", f"https://api.github.com/repos/{full_name}")
    if not data or not isinstance(data, dict):
        return ""
    return (data.get("default_branch") or "").strip()

def list_workflow_runs(gh: GitHubClient, full_name: str, workflow_identifier: str, branch: Optional[str]) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/workflows/{workflow_identifier}/runs"
    params = {"branch": branch} if branch else {}
    return list(gh.paginate(url, params=params, item_key="workflow_runs"))

def list_run_jobs(gh: GitHubClient, full_name: str, run_id: int) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/runs/{run_id}/jobs"
    return list(gh.paginate(url, params={}, item_key="jobs"))


# =========================
# Run timing from jobs window (NEW)
# =========================
def compute_run_window_from_jobs(jobs: List[Dict]) -> Tuple[str, str, Optional[int]]:
    """
    Returns:
      (min_started_at_iso, max_completed_at_iso, duration_seconds)
    """
    if not jobs:
        return "", "", None
    starts: List[datetime] = []
    ends: List[datetime] = []
    for j in jobs:
        sdt = iso_to_dt(j.get("started_at"))
        edt = iso_to_dt(j.get("completed_at"))
        if sdt:
            starts.append(sdt)
        if edt:
            ends.append(edt)
    if not starts or not ends:
        return "", "", None
    smin = min(starts)
    emax = max(ends)
    return (
        smin.isoformat().replace("+00:00", "Z"),
        emax.isoformat().replace("+00:00", "Z"),
        dt_to_seconds(smin, emax),
    )


# =========================
# Instrumentation detection inside runs (jobs/steps) (your existing heuristic)
# =========================
INSTRU_STEP_NAME_RE = re.compile(
    r"(instrument|connected.*androidtest|androidtest|manageddevice|gmd|emulator runner|"
    r"firebase test|test lab|device farm|uiautomator|espresso)",
    re.IGNORECASE,
)
INSTRU_JOB_NAME_RE = re.compile(
    r"(instrument|androidtest|connected|manageddevice|gmd|emulator|firebase|test lab|device farm)",
    re.IGNORECASE,
)

def infer_instru_metrics_from_jobs(jobs: List[Dict], run_created_at: str, run_started_at: str) -> Dict[str, Union[str, int, float, None]]:
    out = {
        "instru_conclusion": "unknown",
        "instru_detect_method": "none",
        "instru_duration_seconds": None,
        "run_duration_seconds": None,
        "runner_labels_union": "",
        "queue_seconds": None,
        "time_to_first_instru_seconds": None,
        "instru_job_count": 0,
        "instru_step_count": 0,
        "instru_job_names": "",
        "instru_step_names": "",
        "instru_total_seconds": None,
        "instru_window_seconds": None,
        "instru_first_started_at": "",
        "instru_last_completed_at": "",
        "instru_share_of_run": None,
    }

    out["queue_seconds"] = dt_to_seconds(iso_to_dt(run_created_at), iso_to_dt(run_started_at))
    if not jobs:
        return out

    starts, ends = [], []
    labels_union: Set[str] = set()
    for j in jobs:
        sdt = iso_to_dt(j.get("started_at"))
        edt = iso_to_dt(j.get("completed_at"))
        if sdt: starts.append(sdt)
        if edt: ends.append(edt)
        for lab in (j.get("labels") or []):
            if isinstance(lab, str) and lab.strip():
                labels_union.add(lab.strip())

    run_dur = dt_to_seconds(min(starts) if starts else None, max(ends) if ends else None)
    out["run_duration_seconds"] = run_dur
    out["runner_labels_union"] = ",".join(sorted(labels_union))

    instru_job_names: List[str] = []
    instru_step_names: List[str] = []
    instru_first_start: Optional[datetime] = None
    instru_last_end: Optional[datetime] = None

    total_seconds = 0
    total_seconds_any = False
    first_match_set = False

    for j in jobs:
        job_name = (j.get("name") or "").strip()
        job_is_instru = bool(INSTRU_JOB_NAME_RE.search(job_name))
        steps = j.get("steps") if isinstance(j.get("steps"), list) else []

        step_matches = []
        for st in steps:
            step_name = (st.get("name") or "").strip()
            if step_name and INSTRU_STEP_NAME_RE.search(step_name):
                step_matches.append(st)

        if job_is_instru or step_matches:
            if job_name:
                instru_job_names.append(job_name)

        for st in step_matches:
            step_name = (st.get("name") or "").strip()
            if step_name:
                instru_step_names.append(step_name)

            sdt = iso_to_dt(st.get("started_at")) or iso_to_dt(j.get("started_at"))
            edt = iso_to_dt(st.get("completed_at")) or iso_to_dt(j.get("completed_at"))

            if sdt and (instru_first_start is None or sdt < instru_first_start):
                instru_first_start = sdt
            if edt and (instru_last_end is None or edt > instru_last_end):
                instru_last_end = edt

            dur = dt_to_seconds(iso_to_dt(st.get("started_at")), iso_to_dt(st.get("completed_at")))
            if dur is None:
                dur = dt_to_seconds(iso_to_dt(j.get("started_at")), iso_to_dt(j.get("completed_at")))
            if dur is not None:
                total_seconds += dur
                total_seconds_any = True

            if not first_match_set:
                out["instru_conclusion"] = (st.get("conclusion") or st.get("status") or "unknown")
                out["instru_detect_method"] = "step"
                out["instru_duration_seconds"] = dur
                first_match_set = True

        if job_is_instru and not step_matches:
            sdt = iso_to_dt(j.get("started_at"))
            edt = iso_to_dt(j.get("completed_at"))

            if sdt and (instru_first_start is None or sdt < instru_first_start):
                instru_first_start = sdt
            if edt and (instru_last_end is None or edt > instru_last_end):
                instru_last_end = edt

            dur = dt_to_seconds(sdt, edt)
            if dur is not None:
                total_seconds += dur
                total_seconds_any = True

            if not first_match_set:
                out["instru_conclusion"] = (j.get("conclusion") or "unknown")
                out["instru_detect_method"] = "job"
                out["instru_duration_seconds"] = dur
                first_match_set = True

    out["instru_job_names"] = safe_join_names(instru_job_names)
    out["instru_step_names"] = safe_join_names(instru_step_names)
    out["instru_job_count"] = len(unique_preserve(instru_job_names))
    out["instru_step_count"] = len(unique_preserve(instru_step_names))

    if total_seconds_any:
        out["instru_total_seconds"] = total_seconds

    if instru_first_start:
        out["instru_first_started_at"] = instru_first_start.isoformat().replace("+00:00", "Z")
        base_start = iso_to_dt(run_started_at) or iso_to_dt(run_created_at)
        out["time_to_first_instru_seconds"] = dt_to_seconds(base_start, instru_first_start)

    if instru_last_end:
        out["instru_last_completed_at"] = instru_last_end.isoformat().replace("+00:00", "Z")

    out["instru_window_seconds"] = dt_to_seconds(instru_first_start, instru_last_end)

    if out["instru_window_seconds"] is not None and run_dur:
        try:
            out["instru_share_of_run"] = round(float(out["instru_window_seconds"]) / float(run_dur), 6)
        except Exception:
            out["instru_share_of_run"] = None

    return out


# =========================
# Read verified workflows
# =========================
def load_verified_workflows(path: Path) -> List[Dict[str, str]]:
    if not path.exists():
        raise FileNotFoundError(f"Verified workflows CSV not found: {path}")
    rows: List[Dict[str, str]] = []
    with path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        rdr = csv.DictReader(f)
        for r in rdr:
            rows.append({k: (v or "") for k, v in r.items()})
    return rows


# =========================
# MAIN
# =========================
def main() -> None:
    if not IN_VERIFIED_WORKFLOWS_CSV.exists():
        raise FileNotFoundError(f"Missing input: {IN_VERIFIED_WORKFLOWS_CSV}")

    if OUT_RUN_INVENTORY_CSV.exists():
        OUT_RUN_INVENTORY_CSV.unlink()

    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH, max_tokens=MAX_TOKENS_TO_USE)
    gh = GitHubClient(tokens)

    rows = load_verified_workflows(IN_VERIFIED_WORKFLOWS_CSV)
    if PROCESS_ONLY_LOOKS_LIKE_INSTRU:
        rows = [r for r in rows if (r.get("looks_like_instru", "").strip().lower() == "yes")]

    if not rows:
        raise RuntimeError("No workflows found to process (check verified CSV or filter).")

    after_dt = iso_to_dt(RUN_CREATED_AT_AFTER) if RUN_CREATED_AT_AFTER else None

    out_fields = [
        "full_name",
        "default_branch",
        "workflow_identifier",
        "workflow_id",
        "workflow_name",
        "workflow_path",

        # workflow-level labels
        "looks_like_instru",
        "gmd_capable",
        "gmd_reasons",
        "styles",
        "invocation_types",
        "evidence_labels",
        "label_source",

        # run
        "run_id",
        "run_number",
        "run_attempt",
        "head_sha",
        "created_at",
        "run_started_at",
        "status",
        "run_conclusion",
        "event",
        "head_branch",
        "html_url",
        "extracted_at_utc",

        # ---- legacy metrics (keep) ----
        "queue_seconds",
        "time_to_first_instru_seconds",
        "instru_conclusion",
        "instru_detect_method",
        "instru_duration_seconds",
        "run_duration_seconds",
        "runner_labels_union",
        "instru_job_count",
        "instru_step_count",
        "instru_job_names",
        "instru_step_names",
        "instru_total_seconds",
        "instru_window_seconds",
        "instru_first_started_at",
        "instru_last_completed_at",
        "instru_share_of_run",

        # ---- NEW S2_ fallback metrics ----
        "S2_run_started_at_jobs_min",
        "S2_run_ended_at_jobs_max",
        "S2_run_duration_seconds_jobs_window",
        "S2_run_timing_source",

        "S2_queue_seconds",
        "S2_time_to_first_instru_seconds",
        "S2_instru_conclusion",
        "S2_instru_detect_method",
        "S2_instru_duration_seconds",
        "S2_run_duration_seconds",
        "S2_runner_labels_union",
        "S2_instru_job_count",
        "S2_instru_step_count",
        "S2_instru_job_names",
        "S2_instru_step_names",
        "S2_instru_total_seconds",
        "S2_instru_window_seconds",
        "S2_instru_first_started_at",
        "S2_instru_last_completed_at",
        "S2_instru_share_of_run",
    ]

    ensure_csv_header(OUT_RUN_INVENTORY_CSV, out_fields)
    existing_run_ids = load_existing_keys(OUT_RUN_INVENTORY_CSV, "run_id")

    default_branch_cache: Dict[str, str] = {}

    wf_iter = rows
    if tqdm is not None:
        wf_iter = tqdm(rows, desc="Stage2: workflows -> runs")

    for wf in wf_iter:
        full_name = (wf.get("full_name") or "").strip()
        workflow_identifier = (wf.get("workflow_identifier") or "").strip()
        workflow_id = (wf.get("workflow_id") or "").strip()
        workflow_name = (wf.get("workflow_name") or "").strip()
        workflow_path = (wf.get("workflow_path") or "").strip()

        if not full_name or not workflow_identifier:
            continue

        if DEFAULT_BRANCH_ONLY:
            if full_name not in default_branch_cache:
                default_branch_cache[full_name] = get_repo_default_branch(gh, full_name)
            default_branch = default_branch_cache[full_name]
            if not default_branch:
                continue
        else:
            default_branch = ""

        branch = default_branch if DEFAULT_BRANCH_ONLY else None

        runs = list_workflow_runs(gh, full_name, workflow_identifier, branch=branch) or []
        if MAX_RUNS_PER_WORKFLOW is not None:
            runs = runs[:MAX_RUNS_PER_WORKFLOW]

        for run in runs:
            run_id = str(run.get("id") or "").strip()
            if not run_id or run_id in existing_run_ids:
                continue

            created_at = run.get("created_at") or ""
            if after_dt:
                cdt = iso_to_dt(created_at)
                if cdt and cdt < after_dt:
                    continue

            head_branch = run.get("head_branch") or ""
            if DEFAULT_BRANCH_ONLY and head_branch and head_branch != default_branch:
                continue

            run_started_at = run.get("run_started_at") or ""
            head_sha = run.get("head_sha") or ""

            jobs = list_run_jobs(gh, full_name, int(run_id)) if FETCH_JOBS_FOR_EACH_RUN else []
            metrics = infer_instru_metrics_from_jobs(
                jobs=jobs,
                run_created_at=created_at,
                run_started_at=run_started_at,
            )

            # NEW: job-based run window (S2_)
            s2_run_start, s2_run_end, s2_run_dur = compute_run_window_from_jobs(jobs)
            s2_run_src = "jobs_window" if s2_run_start and s2_run_end else "missing"

            # ---------- CONSISTENT workflow-label fallback: GMD-only ----------
            wf_gmd_capable = (wf.get("gmd_capable") or "").strip().lower() == "yes"
            wf_inv = (wf.get("invocation_types") or "").lower()
            gmd_workflow_implies_instru = wf_gmd_capable or ("gradle_gmd" in wf_inv)

            if (
                gmd_workflow_implies_instru
                and int(metrics.get("instru_job_count") or 0) == 0
                and int(metrics.get("instru_step_count") or 0) == 0
            ):
                metrics["instru_detect_method"] = "workflow_label_gmd"
                metrics["instru_conclusion"] = (run.get("conclusion") or "unknown")
                if run_started_at:
                    metrics["instru_first_started_at"] = run_started_at
                    metrics["time_to_first_instru_seconds"] = 0
            # -----------------------------------------------------------------

            # Build S2_ mirror for safe fallback usage in Stage 3
            s2 = {
                "S2_queue_seconds": metrics["queue_seconds"],
                "S2_time_to_first_instru_seconds": metrics["time_to_first_instru_seconds"],
                "S2_instru_conclusion": metrics["instru_conclusion"],
                "S2_instru_detect_method": metrics["instru_detect_method"],
                "S2_instru_duration_seconds": metrics["instru_duration_seconds"],
                "S2_run_duration_seconds": metrics["run_duration_seconds"],
                "S2_runner_labels_union": metrics["runner_labels_union"],
                "S2_instru_job_count": metrics["instru_job_count"],
                "S2_instru_step_count": metrics["instru_step_count"],
                "S2_instru_job_names": metrics["instru_job_names"],
                "S2_instru_step_names": metrics["instru_step_names"],
                "S2_instru_total_seconds": metrics["instru_total_seconds"],
                "S2_instru_window_seconds": metrics["instru_window_seconds"],
                "S2_instru_first_started_at": metrics["instru_first_started_at"],
                "S2_instru_last_completed_at": metrics["instru_last_completed_at"],
                "S2_instru_share_of_run": metrics["instru_share_of_run"],
            }

            append_row(OUT_RUN_INVENTORY_CSV, out_fields, {
                "full_name": full_name,
                "default_branch": default_branch,
                "workflow_identifier": workflow_identifier,
                "workflow_id": workflow_id,
                "workflow_name": workflow_name,
                "workflow_path": workflow_path,

                "looks_like_instru": (wf.get("looks_like_instru") or ""),
                "gmd_capable": (wf.get("gmd_capable") or ""),
                "gmd_reasons": (wf.get("gmd_reasons") or ""),
                "styles": (wf.get("styles") or ""),
                "invocation_types": (wf.get("invocation_types") or ""),
                "evidence_labels": (wf.get("evidence_labels") or ""),
                "label_source": (wf.get("label_source") or ""),

                "run_id": run_id,
                "run_number": run.get("run_number") or "",
                "run_attempt": run.get("run_attempt") or "",
                "head_sha": head_sha,
                "created_at": created_at,
                "run_started_at": run_started_at,
                "status": run.get("status") or "",
                "run_conclusion": run.get("conclusion") or "",
                "event": run.get("event") or "",
                "head_branch": head_branch,
                "html_url": run.get("html_url") or "",
                "extracted_at_utc": now_utc_iso(),

                # legacy metrics (stringify as before)
                "queue_seconds": "" if metrics["queue_seconds"] is None else str(metrics["queue_seconds"]),
                "time_to_first_instru_seconds": "" if metrics["time_to_first_instru_seconds"] is None else str(metrics["time_to_first_instru_seconds"]),
                "instru_conclusion": metrics["instru_conclusion"],
                "instru_detect_method": metrics["instru_detect_method"],
                "instru_duration_seconds": "" if metrics["instru_duration_seconds"] is None else str(metrics["instru_duration_seconds"]),
                "run_duration_seconds": "" if metrics["run_duration_seconds"] is None else str(metrics["run_duration_seconds"]),
                "runner_labels_union": metrics["runner_labels_union"],
                "instru_job_count": str(metrics["instru_job_count"]),
                "instru_step_count": str(metrics["instru_step_count"]),
                "instru_job_names": metrics["instru_job_names"],
                "instru_step_names": metrics["instru_step_names"],
                "instru_total_seconds": "" if metrics["instru_total_seconds"] is None else str(metrics["instru_total_seconds"]),
                "instru_window_seconds": "" if metrics["instru_window_seconds"] is None else str(metrics["instru_window_seconds"]),
                "instru_first_started_at": metrics["instru_first_started_at"],
                "instru_last_completed_at": metrics["instru_last_completed_at"],
                "instru_share_of_run": "" if metrics["instru_share_of_run"] is None else str(metrics["instru_share_of_run"]),

                # NEW run timing fallback (S2_)
                "S2_run_started_at_jobs_min": s2_run_start,
                "S2_run_ended_at_jobs_max": s2_run_end,
                "S2_run_duration_seconds_jobs_window": "" if s2_run_dur is None else str(s2_run_dur),
                "S2_run_timing_source": s2_run_src,

                # NEW S2_ mirror metrics (stringify where needed)
                "S2_queue_seconds": "" if s2["S2_queue_seconds"] is None else str(s2["S2_queue_seconds"]),
                "S2_time_to_first_instru_seconds": "" if s2["S2_time_to_first_instru_seconds"] is None else str(s2["S2_time_to_first_instru_seconds"]),
                "S2_instru_conclusion": s2["S2_instru_conclusion"],
                "S2_instru_detect_method": s2["S2_instru_detect_method"],
                "S2_instru_duration_seconds": "" if s2["S2_instru_duration_seconds"] is None else str(s2["S2_instru_duration_seconds"]),
                "S2_run_duration_seconds": "" if s2["S2_run_duration_seconds"] is None else str(s2["S2_run_duration_seconds"]),
                "S2_runner_labels_union": s2["S2_runner_labels_union"],
                "S2_instru_job_count": "" if s2["S2_instru_job_count"] is None else str(s2["S2_instru_job_count"]),
                "S2_instru_step_count": "" if s2["S2_instru_step_count"] is None else str(s2["S2_instru_step_count"]),
                "S2_instru_job_names": s2["S2_instru_job_names"],
                "S2_instru_step_names": s2["S2_instru_step_names"],
                "S2_instru_total_seconds": "" if s2["S2_instru_total_seconds"] is None else str(s2["S2_instru_total_seconds"]),
                "S2_instru_window_seconds": "" if s2["S2_instru_window_seconds"] is None else str(s2["S2_instru_window_seconds"]),
                "S2_instru_first_started_at": s2["S2_instru_first_started_at"],
                "S2_instru_last_completed_at": s2["S2_instru_last_completed_at"],
                "S2_instru_share_of_run": "" if s2["S2_instru_share_of_run"] is None else str(s2["S2_instru_share_of_run"]),
            })

            existing_run_ids.add(run_id)

        time.sleep(SLEEP_BETWEEN_WORKFLOWS_SEC)

    print("Done.")
    print("Wrote:", OUT_RUN_INVENTORY_CSV)


if __name__ == "__main__":
    main()


Stage2: workflows -> runs: 100%|██████████| 374/374 [4:42:29<00:00, 45.32s/it]    

Done.
Wrote: C:\Android Mobile App\ICST2026_Ext\run_inventory.csv


## Stage 3 — Extract step telemetry, derive TTFTS, and enhance run metrics

In [ ]:
# ============================================================
# Stage 3 (UPGRADED): timing model + S2_ fallback integration
#
# Goal:
# - Stage 3 computes the *real* metrics from job/step telemetry when available.
# - If jobs/steps are missing or incomplete, Stage 3 falls back FIRST to Stage2's
#   S2_ fields (derived from the same historical jobs endpoint at Stage 2 time),
#   then to minimal workflow-label proxy if applicable.
#
# Key upgrade:
# - Accept S2_ fallback fields per run, and use them when:
#     * jobs is empty OR
#     * events list is empty OR
#     * anchor/exec windows cannot be derived
#
# IMPORTANT:
# - Stage 3 outputs remain the same metric names (no prefix).
# - Stage 2 fallback is carried alongside (as S2_... columns) so analysts can
#   distinguish measured vs fallback.
# ============================================================

import base64
import csv
import random
import re
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple, Union

import requests

try:
    from tqdm import tqdm
except ImportError:
    tqdm = None

# =========================
# CONFIG
# =========================
TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")
ROOT_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext")

IN_STAGE2_CSV = ROOT_DIR / "run_inventory.csv"

OUT_STAGE3A_RUNS_CSV = ROOT_DIR / "run_metrics_v16_stage3_enhanced.csv"          # 3A
OUT_STAGE3B_STEPS_CSV = ROOT_DIR / "run_steps_v16_stage3_breakdown.csv"          # 3B
OUT_STAGE3C_RUN_PER_STYLE_CSV = ROOT_DIR / "run_per_style_v1_stage3.csv"         # 3C

MAX_TOKENS_TO_USE = 7
PROCESS_ONLY_RELEVANT_ROWS = True

CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 60
MAX_RETRIES_PER_REQUEST = 8
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60
MAX_PAGES_PER_LIST = 2000

FETCH_WORKFLOW_YAML = True
WORKFLOW_YAML_CACHE_MAX = 5000

# =========================
# Helpers
# =========================
BOM = "\ufeff"

def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def iso_to_dt(iso: Optional[str]) -> Optional[datetime]:
    if not iso:
        return None
    try:
        return datetime.fromisoformat(iso.replace("Z", "+00:00"))
    except Exception:
        return None

def dt_to_seconds(a: Optional[datetime], b: Optional[datetime]) -> Optional[int]:
    if not a or not b:
        return None
    try:
        sec = int((b - a).total_seconds())
        return sec if sec >= 0 else None
    except Exception:
        return None

def unique_preserve(seq: Iterable[str]) -> List[str]:
    seen: Set[str] = set()
    out: List[str] = []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def safe_join_names(names: List[str], max_len: int = 800) -> str:
    s = ",".join(unique_preserve([n.strip() for n in names if n and n.strip()]))
    if len(s) <= max_len:
        return s
    return s[: max_len - 3] + "..."

def load_tokens_from_env_file(env_path: Path, max_tokens: int = 3) -> List[str]:
    if not env_path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {env_path}")
    tokens: List[str] = []
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN_") and v:
            tokens.append(v)
            if len(tokens) >= max_tokens:
                break
    if not tokens:
        raise ValueError(f"No tokens found in {env_path}. Expected keys like GITHUB_TOKEN_1=...")
    return tokens

def _clean_key(k: str) -> str:
    return (k or "").replace(BOM, "").strip()

def read_csv_rows(path: Path) -> Tuple[List[Dict[str, str]], List[str]]:
    if not path.exists():
        raise FileNotFoundError(f"Input CSV not found: {path}")
    with path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        rdr = csv.DictReader(f)
        raw_fields = rdr.fieldnames or []
        fields = [_clean_key(x) for x in raw_fields]
        rows: List[Dict[str, str]] = []
        for r in rdr:
            clean_row = {}
            for k, v in r.items():
                ck = _clean_key(k)
                clean_row[ck] = (v or "")
            rows.append(clean_row)
    return rows, fields

def write_csv(path: Path, fieldnames: List[str], rows: List[Dict[str, str]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow(r)

def ensure_csv(path: Path, fieldnames: List[str]) -> None:
    if path.exists():
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()

def append_row(path: Path, fieldnames: List[str], row: Dict[str, str]) -> None:
    with path.open("a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writerow(row)

def split_styles(styles_text: str) -> List[str]:
    return [s.strip() for s in (styles_text or "").split(",") if s.strip()]

# =========================
# GitHub API client
# =========================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None

class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "stage3-derived-3c/1.1",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        now = int(time.time())
        candidates = []
        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                candidates.append((0, i))
            else:
                if st.reset_epoch is not None and st.reset_epoch <= now:
                    candidates.append((0, i))
                else:
                    candidates.append((1, i))
        candidates.sort()
        return candidates[0][1]

    def _sleep_until_reset(self) -> None:
        now = int(time.time())
        resets = [st.reset_epoch for st in self.tokens if st.reset_epoch]
        if not resets:
            time.sleep(5)
            return
        soonest = min(resets)
        sleep_s = max(1, soonest - now + 2)
        print(f"[rate-limit] sleeping {sleep_s}s until reset...")
        time.sleep(sleep_s)

    def _backoff(self, attempt: int) -> None:
        sleep_s = min(BACKOFF_CAP_S, (BACKOFF_BASE_S ** attempt)) + random.random()
        time.sleep(sleep_s)

    def request_json(self, method: str, url: str, params: Optional[Dict] = None) -> Optional[Union[Dict, List]]:
        last_status = None
        for attempt in range(1, MAX_RETRIES_PER_REQUEST + 1):
            idx = self._pick_idx()
            st = self.tokens[idx]
            self.session.headers["Authorization"] = f"Bearer {st.token}"
            try:
                resp = self.session.request(method, url, params=params, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            except requests.exceptions.RequestException:
                self._backoff(attempt)
                continue

            last_status = resp.status_code

            rem = resp.headers.get("X-RateLimit-Remaining")
            if rem is not None:
                try:
                    st.remaining = int(rem)
                except Exception:
                    pass
            rst = resp.headers.get("X-RateLimit-Reset")
            if rst is not None:
                try:
                    st.reset_epoch = int(rst)
                except Exception:
                    pass

            if resp.status_code == 404:
                return None

            retry_after = resp.headers.get("Retry-After")
            if resp.status_code in (403, 429) and retry_after:
                try:
                    ra = int(retry_after)
                    time.sleep(min(BACKOFF_CAP_S, max(1, ra)) + random.random())
                    continue
                except Exception:
                    pass

            text_l = (resp.text or "").lower()
            if resp.status_code in (403, 429) and (
                "rate limit" in text_l
                or "secondary rate limit" in text_l
                or "abuse detection" in text_l
                or "too many requests" in text_l
            ):
                if any((t.remaining is None) or (t.remaining > 0) for t in self.tokens):
                    self._backoff(attempt)
                    continue
                self._sleep_until_reset()
                continue

            if resp.status_code in (500, 502, 503, 504):
                self._backoff(attempt)
                continue

            if resp.status_code >= 400:
                return None

            try:
                return resp.json()
            except Exception:
                return None

        print(f"[giveup] {method} {url} after {MAX_RETRIES_PER_REQUEST} tries (last_status={last_status})")
        return None

    def paginate(self, url: str, params: Optional[Dict], item_key: str) -> Iterable[Dict]:
        page = 1
        while page <= MAX_PAGES_PER_LIST:
            p = dict(params or {})
            p.update({"per_page": 100, "page": page})
            data = self.request_json("GET", url, params=p)
            if data is None:
                return
            items = data if isinstance(data, list) else data.get(item_key, [])
            if not items:
                return
            for it in items:
                yield it
            if isinstance(items, list) and len(items) < 100:
                return
            page += 1

# =========================
# GitHub endpoints
# =========================
def list_run_jobs(gh: GitHubClient, full_name: str, run_id: int) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/runs/{run_id}/jobs"
    return list(gh.paginate(url, params={}, item_key="jobs"))

def fetch_workflow_yaml(gh: GitHubClient, full_name: str, workflow_path: str, ref: str) -> str:
    url = f"https://api.github.com/repos/{full_name}/contents/{workflow_path.lstrip('/')}"
    data = gh.request_json("GET", url, params={"ref": ref})
    if not data or not isinstance(data, dict):
        return ""
    if data.get("encoding") == "base64" and data.get("content"):
        try:
            return base64.b64decode(data["content"]).decode("utf-8", errors="ignore")
        except Exception:
            return ""
    dl = data.get("download_url")
    if dl:
        try:
            r = gh.session.get(dl, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            if r.status_code == 200:
                return r.text or ""
        except requests.exceptions.RequestException:
            return ""
    return ""

# =========================
# Patterns (same as your updated Stage 3)
# =========================
EXPLICIT_INSTRU_RE = re.compile(
    r"(?is)\b("
    r"adb\s+shell\s+am\s+instrument|"
    r"\bconnected\w*androidtest\b|\bconnectedcheck\b|\bdevicecheck\b|\balldevicescheck\b|"
    r"\bmanageddevice\w*check\b|\bmanageddevice\w*androidtest\b|"
    r"\bfirebase\s+test\s+android\s+run\b|\bgcloud\b.*\bfirebase\s+test\b.*\bandroid\b.*\brun\b|"
    r"\bflank\s+android\s+run\b|"
    r"\bappcenter\s+test\b|"
    r"\bsaucectl\b|\bbrowserstack\b|\bbstack\b|"
    r"\bmaestro\s+cloud\b|"
    r"\bemulator\.wtf\b"
    r")\b"
)

GMD_SETUP_TASK_RE = re.compile(
    r"(?is)\b(\./gradlew\b|gradle\s+|gradlew\.bat\b)[^\n\r]*\b("
    r":[\w:-]*api\d+setup|"
    r":[\w:-]*pixel[\w-]*api\d+setup|"
    r"manageddevice[\w-]*setup|"
    r"pixel[\w-]*api\d+setup"
    r")\b"
)

GMD_LIFECYCLE_TASK_RE = re.compile(
    r"(?is)\b(\./gradlew\b|gradle\s+|gradlew\.bat\b)[^\n\r]*\b("
    r"manageddevice[\w:-]*(check|androidtest|test)|"
    r":[\w:-]*manageddevice[\w:-]*(check|androidtest|test)"
    r")\b"
)

BASELINE_PROFILE_RE = re.compile(
    r"(?is)\b(\./gradlew\b|gradle\s+|gradlew\.bat\b)[^\n\r]*\b("
    r"generatebaselineprofile|"
    r"baselineprofile"
    r")\b"
)

THIRD_PARTY_ANY_RE = re.compile(
    r"(gcloud.*firebase\s+test\s+android\s+run|firebase\s+test\s+lab|flank\s+android\s+run|"
    r"appcenter\s+test\s+run\s+android|saucectl|browserstack|bstack|maestro\s+cloud|"
    r"emulator\.wtf)",
    re.IGNORECASE,
)

EMU_COMMUNITY_ACTION_RE = re.compile(
    r"(reactivecircus/android-emulator-runner|android-emulator-runner)",
    re.IGNORECASE,
)

EMU_CUSTOM_SCRIPT_RE = re.compile(
    r"(?is)\b(avdmanager|sdkmanager|emulator\b|start[-_ ]emulator|adb\s+wait[- ]?for[- ]?device)\b"
)

REAL_DEVICE_ADB_RE = re.compile(r"(?mi)\badb\s+-s\s+(?!emulator-\d+\b)(?!localhost:\d+\b)(?!127\.0\.0\.1:\d+\b)\S+\b")
ENV_ANY_RE = re.compile(
    r"(reactivecircus|android-emulator-runner|emulator\b|avd\b|avdmanager|sdkmanager|"
    r"start[-_ ]emulator|android-wait-for-emulator|adb\s+wait[- ]?for[- ]?device|kvm|"
    r"android-actions/setup-android)",
    re.IGNORECASE,
)
ARTIFACT_RE = re.compile(r"(upload[- ]artifact|actions/upload-artifact)", re.IGNORECASE)
GRADLE_CMD_RE = re.compile(r"(?mi)\b(\./gradlew\b|gradlew\.bat\b|gradle\s+)\b")

# =========================
# YAML step extraction (same as your version)
# =========================
def _count_leading_spaces(s: str) -> int:
    return len(s) - len(s.lstrip(" "))

def parse_workflow_steps(yaml_text: str) -> Dict[str, Dict[str, str]]:
    out: Dict[str, Dict[str, str]] = {}
    if not yaml_text:
        return out

    lines = yaml_text.splitlines()
    n = len(lines)
    i = 0

    while i < n:
        line = lines[i]
        m = re.match(r"^(\s*)-\s*name\s*:\s*(.+?)\s*$", line)
        if not m:
            i += 1
            continue

        base_indent = len(m.group(1))
        step_name = m.group(2).strip().strip('"').strip("'")
        key = step_name.lower()

        j = i + 1
        block_lines = [line]
        while j < n:
            nxt = lines[j]
            m2 = re.match(r"^(\s*)-\s*name\s*:\s*(.+?)\s*$", nxt)
            if m2 and len(m2.group(1)) == base_indent:
                break
            block_lines.append(nxt)
            j += 1

        block = "\n".join(block_lines)

        uses_val = ""
        m_uses = re.search(r"(?mi)^\s*uses\s*:\s*([^\n\r#]+)", block)
        if m_uses:
            uses_val = m_uses.group(1).strip().strip('"').strip("'")

        run_val = ""
        m_run = re.search(r"(?mi)^\s*run\s*:\s*(.*)$", block)
        if m_run:
            run_line_text = m_run.group(0)
            run_start_idx = None
            for idx, bl in enumerate(block_lines):
                if bl.strip() == run_line_text.strip():
                    run_start_idx = idx
                    break
            if run_start_idx is not None:
                run_indent = _count_leading_spaces(block_lines[run_start_idx])
                rhs = block_lines[run_start_idx].split("run:", 1)[1].strip()
                if rhs in ("|", ">"):
                    k = run_start_idx + 1
                    acc = []
                    while k < len(block_lines):
                        l = block_lines[k]
                        if l.strip() == "":
                            acc.append("")
                            k += 1
                            continue
                        if _count_leading_spaces(l) <= run_indent:
                            break
                        acc.append(l.strip("\n"))
                        k += 1
                    run_val = "\n".join(acc).strip()
                else:
                    run_val = rhs.strip()

        with_script = ""
        with_line = None
        for idx, bl in enumerate(block_lines):
            if re.match(r"^\s*with\s*:\s*$", bl):
                with_line = idx
                break
        if with_line is not None:
            with_indent = _count_leading_spaces(block_lines[with_line])
            k = with_line + 1
            while k < len(block_lines):
                l = block_lines[k]
                if l.strip() == "":
                    k += 1
                    continue
                if _count_leading_spaces(l) <= with_indent:
                    break
                m_script = re.match(r"^\s*script\s*:\s*(.*)\s*$", l)
                if m_script:
                    rhs = m_script.group(1).strip()
                    script_indent = _count_leading_spaces(l)
                    if rhs in ("|", ">"):
                        kk = k + 1
                        acc = []
                        while kk < len(block_lines):
                            ll = block_lines[kk]
                            if ll.strip() == "":
                                acc.append("")
                                kk += 1
                                continue
                            if _count_leading_spaces(ll) <= script_indent:
                                break
                            acc.append(ll.strip("\n"))
                            kk += 1
                        with_script = "\n".join(acc).strip()
                    else:
                        with_script = rhs
                    break
                k += 1

        out[key] = {"run": run_val or "", "uses": uses_val or "", "with_script": with_script or "", "blob": block}
        i = j

    return out

# =========================
# Step classification + style tagging (your fixed Emu_Community logic)
# =========================
def classify_step(step_name: str, y: Optional[Dict[str, str]]) -> Dict[str, bool]:
    y = y or {}
    run_txt = y.get("run", "") or ""
    uses_txt = y.get("uses", "") or ""
    script_txt = y.get("with_script", "") or ""
    blob = y.get("blob", "") or ""

    combined = "\n".join([step_name or "", run_txt, uses_txt, script_txt, blob])

    is_third_party = bool(THIRD_PARTY_ANY_RE.search(combined))
    is_explicit_instru = bool(EXPLICIT_INSTRU_RE.search(combined))
    is_gmd_setup = bool(GMD_SETUP_TASK_RE.search(combined)) or bool(re.search(r"(?i)\bsetup\s+gmd\b", step_name or ""))
    is_gmd_lifecycle_task = bool(GMD_LIFECYCLE_TASK_RE.search(combined))
    is_baseline_profile = bool(BASELINE_PROFILE_RE.search(combined)) or bool(re.search(r"(?i)\bbaseline\s*profile\b", step_name or ""))
    is_env = bool(ENV_ANY_RE.search(combined))
    is_gradle = bool(GRADLE_CMD_RE.search(combined))
    is_artifact = bool(ARTIFACT_RE.search(combined))
    is_emu_community_action = bool(EMU_COMMUNITY_ACTION_RE.search(combined))
    is_emu_custom_script = bool(EMU_CUSTOM_SCRIPT_RE.search(combined)) and not is_emu_community_action
    is_real_device = bool(REAL_DEVICE_ADB_RE.search(combined))

    return {
        "explicit_instru": is_explicit_instru,
        "third_party": is_third_party,
        "gmd_setup": is_gmd_setup,
        "gmd_lifecycle_task": is_gmd_lifecycle_task,
        "baseline_profile": is_baseline_profile,
        "env_setup": is_env,
        "gradle": is_gradle,
        "artifact": is_artifact,
        "emu_community_action": is_emu_community_action,
        "emu_custom_script": is_emu_custom_script,
        "real_device": is_real_device,
    }

def tag_step_style(flags: Dict[str, bool], styles_text: str, run_has_gmd_evidence: bool) -> Tuple[str, str]:
    styles_l = (styles_text or "").lower()
    has_comm = "emu_community" in styles_l
    has_custom = "emu_custom" in styles_l

    if flags.get("third_party"):
        return "Third-Party", "third_party_signal"
    if flags.get("real_device"):
        return "Real-Device", "real_device_signal"
    if flags.get("gmd_setup"):
        return "GMD", "gmd_setup_signal"
    if flags.get("gmd_lifecycle_task"):
        return "GMD", "gmd_lifecycle_task_signal"
    if flags.get("baseline_profile") and ("gmd" in styles_l) and run_has_gmd_evidence:
        return "GMD", "baseline_profile_with_gmd_evidence"
    if flags.get("emu_community_action"):
        return "Emu_Community", "emu_runner_action"

    if flags.get("emu_custom_script"):
        if has_custom and not has_comm:
            return "Emu_Custom", "scripted_emulator_signal"
        if has_comm and not has_custom:
            return "Emu_Community", "scripted_emulator_signal_under_community_run"
        return "Emu_Custom", "scripted_emulator_signal"

    if flags.get("explicit_instru"):
        if has_comm and not has_custom:
            return "Emu_Community", "explicit_instru_with_run_style_hint"
        if has_custom and not has_comm:
            return "Emu_Custom", "explicit_instru_with_run_style_hint"
        if "gmd" in styles_l:
            return "GMD", "explicit_instru_with_run_style_hint"
        return "", ""

    return "", ""

# =========================
# Anchor + exec step rules (as your updated timing model)
# =========================
def is_exec_step(flags: Dict[str, bool]) -> bool:
    return bool(
        flags.get("explicit_instru")
        or flags.get("third_party")
        or flags.get("gmd_lifecycle_task")
        or flags.get("emu_community_action")
    )

def pick_instru_anchor_from_candidates(
    cands: List[Tuple[datetime, str, Dict[str, bool]]]
) -> Tuple[Optional[datetime], str, str]:
    if not cands:
        return None, "", "missing"

    tier1 = [(t, n) for (t, n, f) in cands if f.get("explicit_instru") or f.get("third_party")]
    if tier1:
        tier1.sort(key=lambda x: x[0])
        return tier1[0][0], tier1[0][1], "explicit_instru_step"

    tier2 = [(t, n) for (t, n, f) in cands if f.get("gmd_setup") or f.get("gmd_lifecycle_task")]
    if tier2:
        tier2.sort(key=lambda x: x[0])
        return tier2[0][0], tier2[0][1], "gmd_setup_step"

    tier3 = [(t, n) for (t, n, f) in cands if f.get("emu_community_action")]
    if tier3:
        tier3.sort(key=lambda x: x[0])
        return tier3[0][0], tier3[0][1], "emu_runner_action_step"

    tier4 = [(t, n) for (t, n, f) in cands if f.get("emu_custom_script")]
    if tier4:
        tier4.sort(key=lambda x: x[0])
        return tier4[0][0], tier4[0][1], "scripted_emulator_step"

    tier5 = [(t, n) for (t, n, f) in cands if f.get("baseline_profile")]
    if tier5:
        tier5.sort(key=lambda x: x[0])
        return tier5[0][0], tier5[0][1], "baseline_profile_step"

    return None, "", "missing"

# =========================
# Metric computation (UPDATED) + S2_ fallback
# =========================
def compute_metrics_for_event_set(
    base_start: Optional[datetime],
    events: List[Tuple[Optional[datetime], Optional[datetime], Optional[int], str, Dict[str, bool]]],
    stage2_instru_detect_method: str,
    s2_fallback: Dict[str, str],
) -> Dict[str, Union[str, int, None]]:
    """
    Stage3 primary: compute from events (steps).
    Stage3 fallback: use S2_ fields when events are missing.
    """
    out: Dict[str, Union[str, int, None]] = {
        "first_test_step_started_at": "",
        "ttfts_seconds": None,
        "ttfts_source": "missing",

        "instru_started_at": "",
        "instru_ended_at": "",
        "test_exec_started_at": "",
        "test_exec_ended_at": "",

        "instru_duration_seconds": None,
        "pre_test_overhead_seconds": None,
        "core_instru_window_seconds": None,
        "post_test_overhead_seconds": None,
        "instru_exec_sum_seconds": None,
        "instru_exec_window_seconds": None,
        "instru_exec_step_count": None,

        "env_setup_sum_seconds": None,
        "artifact_sum_seconds": None,
    }

    # ---------- if no events: FIRST fallback to Stage2 S2_ ----------
    if not events:
        s2_first = (s2_fallback.get("S2_instru_first_started_at") or "").strip()
        s2_last = (s2_fallback.get("S2_instru_last_completed_at") or "").strip()
        s2_window = (s2_fallback.get("S2_instru_window_seconds") or "").strip()
        s2_ttfi = (s2_fallback.get("S2_time_to_first_instru_seconds") or "").strip()

        if s2_first:
            out["instru_started_at"] = s2_first
            out["first_test_step_started_at"] = s2_first
            if base_start and not s2_ttfi:
                out["ttfts_seconds"] = dt_to_seconds(base_start, iso_to_dt(s2_first))
                out["ttfts_source"] = "S2_instru_first_started_at"
            elif s2_ttfi:
                try:
                    out["ttfts_seconds"] = int(float(s2_ttfi))
                    out["ttfts_source"] = "S2_time_to_first_instru_seconds"
                except Exception:
                    out["ttfts_source"] = "S2_instru_first_started_at"

        if s2_last:
            out["instru_ended_at"] = s2_last

        if s2_window:
            try:
                out["instru_duration_seconds"] = int(float(s2_window))
            except Exception:
                pass

        # If no S2_ either: keep your workflow_label proxy behavior
        if not s2_first and (stage2_instru_detect_method or "").strip().lower().startswith("workflow_label"):
            out["ttfts_seconds"] = 0
            out["ttfts_source"] = "workflow_label_proxy"

        return out

    # ---------- primary computation from steps ----------
    total_env = 0
    total_art = 0

    exec_first: Optional[datetime] = None
    exec_last: Optional[datetime] = None
    exec_sum = 0
    exec_count = 0

    cands: List[Tuple[datetime, str, Dict[str, bool]]] = []
    latest_any_end_after_exec: Optional[datetime] = None

    for (st_start, st_end, st_dur, step_name, flags) in events:
        if flags.get("env_setup") and st_dur is not None:
            total_env += st_dur
        if flags.get("artifact") and st_dur is not None:
            total_art += st_dur

        if st_start:
            cands.append((st_start, step_name, flags))

        if is_exec_step(flags) and st_start and st_end:
            if exec_first is None or st_start < exec_first:
                exec_first = st_start
            if exec_last is None or st_end > exec_last:
                exec_last = st_end
            if st_dur is not None:
                exec_sum += st_dur
            exec_count += 1

        if exec_first and st_end:
            if st_start and st_start >= exec_first:
                if latest_any_end_after_exec is None or st_end > latest_any_end_after_exec:
                    latest_any_end_after_exec = st_end

    out["env_setup_sum_seconds"] = total_env if total_env > 0 else None
    out["artifact_sum_seconds"] = total_art if total_art > 0 else None

    anchor_dt, _anchor_name, anchor_source = pick_instru_anchor_from_candidates(cands)
    if anchor_dt is None:
        fallback = [(t, n, f) for (t, n, f) in cands if f.get("env_setup") or f.get("gradle")]
        if fallback:
            fallback.sort(key=lambda x: x[0])
            anchor_dt = fallback[0][0]
            anchor_source = "fallback_env_or_gradle_step"

    if anchor_dt and base_start:
        out["instru_started_at"] = anchor_dt.isoformat().replace("+00:00", "Z")
        out["first_test_step_started_at"] = out["instru_started_at"]
        out["ttfts_seconds"] = dt_to_seconds(base_start, anchor_dt)
        out["ttfts_source"] = anchor_source
    else:
        if (stage2_instru_detect_method or "").strip().lower().startswith("workflow_label"):
            out["ttfts_seconds"] = 0
            out["ttfts_source"] = "workflow_label_proxy"

    if exec_first:
        out["test_exec_started_at"] = exec_first.isoformat().replace("+00:00", "Z")
    if exec_last:
        out["test_exec_ended_at"] = exec_last.isoformat().replace("+00:00", "Z")

    instru_end: Optional[datetime] = None
    if latest_any_end_after_exec:
        instru_end = latest_any_end_after_exec
    elif exec_last:
        instru_end = exec_last

    if instru_end:
        out["instru_ended_at"] = instru_end.isoformat().replace("+00:00", "Z")

    if anchor_dt and instru_end:
        out["instru_duration_seconds"] = dt_to_seconds(anchor_dt, instru_end)

    if anchor_dt and exec_first:
        out["pre_test_overhead_seconds"] = dt_to_seconds(anchor_dt, exec_first)

    if exec_first and exec_last:
        out["core_instru_window_seconds"] = dt_to_seconds(exec_first, exec_last)
        out["instru_exec_window_seconds"] = out["core_instru_window_seconds"]
        out["instru_exec_sum_seconds"] = exec_sum if exec_sum > 0 else None
        out["instru_exec_step_count"] = exec_count if exec_count > 0 else None

    if exec_last and instru_end:
        out["post_test_overhead_seconds"] = dt_to_seconds(exec_last, instru_end)

    return out

# =========================
# Stage 3 builders
# =========================
STYLE_METRIC_KEYS = [
    "first_test_step_started_at",
    "ttfts_seconds",
    "ttfts_source",

    "instru_started_at",
    "instru_ended_at",
    "test_exec_started_at",
    "test_exec_ended_at",

    "instru_duration_seconds",
    "pre_test_overhead_seconds",
    "core_instru_window_seconds",
    "post_test_overhead_seconds",
    "instru_exec_sum_seconds",
    "instru_exec_window_seconds",
    "instru_exec_step_count",

    "env_setup_sum_seconds",
    "artifact_sum_seconds",
]

def build_stage3_outputs_for_run(
    jobs: List[Dict],
    run_created_at: str,
    run_started_at: str,
    yaml_steps: Dict[str, Dict[str, str]],
    styles_text: str,
    stage2_instru_detect_method: str,
    s2_fallback: Dict[str, str],
    full_name: str,
    run_id: str,
    workflow_identifier: str,
    workflow_path: str,
    head_sha: str,
) -> Tuple[Dict[str, Union[str, int, float, None]], List[Dict[str, str]], List[Dict[str, str]]]:

    run_metrics: Dict[str, Union[str, int, float, None]] = {k: ("" if k.endswith("_at") else None) for k in STYLE_METRIC_KEYS}
    run_metrics.update({
        "ttfts_source": "missing",
        "third_party_job_count": 0,
        "third_party_job_names": "",
    })

    step_rows: List[Dict[str, str]] = []
    per_style_rows: List[Dict[str, str]] = []

    base_start = iso_to_dt(run_started_at) or iso_to_dt(run_created_at)

    # ---------- no jobs: still allow S2_ fallback ----------
    if not jobs:
        baseline = compute_metrics_for_event_set(base_start, [], stage2_instru_detect_method, s2_fallback)
        for k in STYLE_METRIC_KEYS:
            run_metrics[k] = baseline.get(k)

        declared = split_styles(styles_text) or [""]
        for s in declared:
            row = {"style": s}
            for k in STYLE_METRIC_KEYS:
                v = run_metrics.get(k)
                row[k] = "" if v is None else str(v)
            per_style_rows.append(row)

        return run_metrics, step_rows, per_style_rows

    # PASS 1: collect steps
    tmp_steps: List[Tuple[str, str, Optional[datetime], Optional[datetime], Optional[int], Dict[str, bool], str, str]] = []
    third_party_job_names: List[str] = []
    third_party_count = 0

    for j in jobs:
        job_id = str(j.get("id") or "")
        job_name = (j.get("name") or "").strip()
        steps = j.get("steps") if isinstance(j.get("steps"), list) else []

        job_start = iso_to_dt(j.get("started_at"))
        job_end = iso_to_dt(j.get("completed_at"))

        job_is_3p = bool(THIRD_PARTY_ANY_RE.search(job_name)) or any(
            THIRD_PARTY_ANY_RE.search((st.get("name") or "")) for st in steps
        )
        if job_is_3p:
            third_party_count += 1
            if job_name:
                third_party_job_names.append(job_name)

        for st in steps:
            step_name = (st.get("name") or "").strip()
            if not step_name:
                continue

            st_start = iso_to_dt(st.get("started_at")) or job_start
            st_end = iso_to_dt(st.get("completed_at")) or job_end

            y = yaml_steps.get(step_name.lower())
            flags = classify_step(step_name, y)

            if job_is_3p:
                flags["third_party"] = True
                flags["explicit_instru"] = True

            st_dur = dt_to_seconds(iso_to_dt(st.get("started_at")), iso_to_dt(st.get("completed_at")))
            if st_dur is None:
                st_dur = dt_to_seconds(job_start, job_end)

            if flags["artifact"]:
                cat = "artifact"
            elif flags["explicit_instru"] or flags["third_party"] or flags["baseline_profile"] or flags["emu_community_action"]:
                cat = "test"
            elif flags["gmd_setup"] or flags["gmd_lifecycle_task"]:
                cat = "gmd_setup"
            elif flags["env_setup"]:
                cat = "env_setup"
            elif flags["gradle"]:
                cat = "gradle"
            else:
                cat = "other"

            tmp_steps.append((job_id, job_name, st_start, st_end, st_dur, flags, step_name, cat))

    run_has_gmd_evidence = any(
        (flags.get("gmd_setup") or flags.get("gmd_lifecycle_task"))
        for (_job_id, _job_name, _st_start, _st_end, _st_dur, flags, _step_name, _cat) in tmp_steps
    )

    # PASS 2: tag steps + group by style
    all_events: List[Tuple[Optional[datetime], Optional[datetime], Optional[int], str, Dict[str, bool]]] = []
    events_by_style: Dict[str, List[Tuple[Optional[datetime], Optional[datetime], Optional[int], str, Dict[str, bool]]]] = {}

    for (job_id, job_name, st_start, st_end, st_dur, flags, step_name, cat) in tmp_steps:
        step_style_tag, step_style_reason = tag_step_style(flags, styles_text, run_has_gmd_evidence)

        step_rows.append({
            "full_name": full_name,
            "run_id": run_id,
            "workflow_identifier": workflow_identifier,
            "workflow_path": workflow_path,
            "head_sha": head_sha,
            "styles": styles_text,
            "job_id": job_id,
            "job_name": job_name,
            "step_name": step_name,
            "category": cat,
            "step_style_tag": step_style_tag,
            "step_style_reason": step_style_reason,
            "started_at": st_start.isoformat() if st_start else "",
            "completed_at": st_end.isoformat() if st_end else "",
            "duration_seconds": "" if st_dur is None else str(st_dur),
        })

        ev = (st_start, st_end, st_dur, step_name, flags)
        all_events.append(ev)

        if step_style_tag:
            events_by_style.setdefault(step_style_tag, []).append(ev)

    # 3A baseline metrics (fresh, but fallback-aware internally)
    baseline = compute_metrics_for_event_set(base_start, all_events, stage2_instru_detect_method, s2_fallback)
    for k in STYLE_METRIC_KEYS:
        run_metrics[k] = baseline.get(k)

    run_metrics["third_party_job_count"] = third_party_count
    run_metrics["third_party_job_names"] = safe_join_names(third_party_job_names)

    # 3C derived per-style
    declared_styles = split_styles(styles_text) or [""]
    is_multi_style = len(declared_styles) > 1

    if not is_multi_style:
        s = declared_styles[0]
        row = {"style": s}
        for k in STYLE_METRIC_KEYS:
            v = run_metrics.get(k)
            row[k] = "" if v is None else str(v)
        per_style_rows.append(row)
        return run_metrics, step_rows, per_style_rows

    for s in declared_styles:
        row = {"style": s}
        for k in STYLE_METRIC_KEYS:
            v = run_metrics.get(k)
            row[k] = "" if v is None else str(v)

        s_events = events_by_style.get(s, [])
        if s_events:
            s_metrics = compute_metrics_for_event_set(base_start, s_events, stage2_instru_detect_method, s2_fallback)
            for k in STYLE_METRIC_KEYS:
                v = s_metrics.get(k)
                row[k] = "" if v is None else str(v)

        per_style_rows.append(row)

    return run_metrics, step_rows, per_style_rows

# =========================
# MAIN
# =========================
def main() -> None:
    for p in [OUT_STAGE3A_RUNS_CSV, OUT_STAGE3B_STEPS_CSV, OUT_STAGE3C_RUN_PER_STYLE_CSV]:
        if p.exists():
            p.unlink()

    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH, max_tokens=MAX_TOKENS_TO_USE)
    gh = GitHubClient(tokens)

    rows, in_fields = read_csv_rows(IN_STAGE2_CSV)
    if not rows:
        raise RuntimeError("No rows in Stage-2 input CSV.")

    if PROCESS_ONLY_RELEVANT_ROWS:
        def is_relevant(r: Dict[str, str]) -> bool:
            det = (r.get("instru_detect_method", "") or "").strip().lower()
            styles = (r.get("styles", "") or "").lower()
            inv = (r.get("invocation_types", "") or "").lower()
            looks = (r.get("looks_like_instru", "") or "").strip().lower()
            return (
                looks == "yes"
                or det not in ("", "none", "unknown")
                or ("third-party" in styles)
                or ("gmd" in styles)
                or ("emu_custom" in styles)
                or ("emu_community" in styles)
                or ("3p" in inv)
            )
        target = [r for r in rows if is_relevant(r)]
    else:
        target = rows

    print(f"[Stage3] Rows total: {len(rows)} | Rows to enhance: {len(target)}")

    new_cols_3a = (
        STYLE_METRIC_KEYS
        + [
            "third_party_job_count",
            "third_party_job_names",
            "stage3_extracted_at_utc",
        ]
    )

    out_fieldnames_3a = list(in_fields)
    for c in new_cols_3a:
        if c not in out_fieldnames_3a:
            out_fieldnames_3a.append(c)

    steps_fields = [
        "full_name",
        "run_id",
        "workflow_identifier",
        "workflow_path",
        "head_sha",
        "styles",
        "job_id",
        "job_name",
        "step_name",
        "category",
        "step_style_tag",
        "step_style_reason",
        "started_at",
        "completed_at",
        "duration_seconds",
        "stage3_extracted_at_utc",
    ]
    ensure_csv(OUT_STAGE3B_STEPS_CSV, steps_fields)

    per_style_fields = list(out_fieldnames_3a)
    if "styles" in per_style_fields:
        i = per_style_fields.index("styles") + 1
        per_style_fields.insert(i, "style")
    else:
        per_style_fields.append("style")
    ensure_csv(OUT_STAGE3C_RUN_PER_STYLE_CSV, per_style_fields)

    yaml_cache: Dict[Tuple[str, str, str], Dict[str, Dict[str, str]]] = {}

    it = target
    if tqdm is not None:
        it = tqdm(target, desc="Stage3: build 3A/3B/3C")

    all_per_style_rows: List[Dict[str, str]] = []

    for r in it:
        full_name = (r.get("full_name") or "").strip()
        run_id = (r.get("run_id") or "").strip()
        created_at = r.get("created_at") or ""
        run_started_at = r.get("run_started_at") or ""
        workflow_identifier = (r.get("workflow_identifier") or "").strip()
        workflow_path = (r.get("workflow_path") or "").strip()
        head_sha = (r.get("head_sha") or "").strip()
        styles_text = (r.get("styles") or "")
        stage2_det = (r.get("instru_detect_method") or "")

        if not full_name or not run_id:
            continue

        # Collect S2_ fallback bundle once
        s2_fallback = {
            "S2_instru_first_started_at": r.get("S2_instru_first_started_at", ""),
            "S2_instru_last_completed_at": r.get("S2_instru_last_completed_at", ""),
            "S2_instru_window_seconds": r.get("S2_instru_window_seconds", ""),
            "S2_time_to_first_instru_seconds": r.get("S2_time_to_first_instru_seconds", ""),
        }

        yaml_steps: Dict[str, Dict[str, str]] = {}
        if FETCH_WORKFLOW_YAML and workflow_path and head_sha:
            ck = (full_name, workflow_path, head_sha)
            if ck in yaml_cache:
                yaml_steps = yaml_cache[ck]
            else:
                yml = fetch_workflow_yaml(gh, full_name, workflow_path, ref=head_sha)
                yaml_steps = parse_workflow_steps(yml)
                if len(yaml_cache) < WORKFLOW_YAML_CACHE_MAX:
                    yaml_cache[ck] = yaml_steps

        jobs = list_run_jobs(gh, full_name, int(run_id)) or []

        run_metrics, step_rows, per_style_rows = build_stage3_outputs_for_run(
            jobs=jobs,
            run_created_at=created_at,
            run_started_at=run_started_at,
            yaml_steps=yaml_steps,
            styles_text=styles_text,
            stage2_instru_detect_method=stage2_det,
            s2_fallback=s2_fallback,
            full_name=full_name,
            run_id=run_id,
            workflow_identifier=workflow_identifier,
            workflow_path=workflow_path,
            head_sha=head_sha,
        )

        extracted_ts = now_utc_iso()
        r["stage3_extracted_at_utc"] = extracted_ts

        for k in new_cols_3a:
            if k == "stage3_extracted_at_utc":
                continue
            v = run_metrics.get(k)
            r[k] = "" if v is None else str(v)

        for sr in step_rows:
            sr2 = dict(sr)
            sr2["stage3_extracted_at_utc"] = extracted_ts
            append_row(OUT_STAGE3B_STEPS_CSV, steps_fields, sr2)

        for pr in per_style_rows:
            pr2 = dict(r)
            pr2["style"] = pr.get("style", "")
            for k in STYLE_METRIC_KEYS:
                if k in pr:
                    pr2[k] = pr[k]
            pr2["stage3_extracted_at_utc"] = extracted_ts
            all_per_style_rows.append(pr2)

    write_csv(OUT_STAGE3A_RUNS_CSV, out_fieldnames_3a, rows)
    write_csv(OUT_STAGE3C_RUN_PER_STYLE_CSV, per_style_fields, all_per_style_rows)

    print("[done] 3A run metrics:", OUT_STAGE3A_RUNS_CSV)
    print("[done] 3B step breakdown:", OUT_STAGE3B_STEPS_CSV)
    print("[done] 3C run×style:", OUT_STAGE3C_RUN_PER_STYLE_CSV)

if __name__ == "__main__":
    main()


[Stage3] Rows total: 32777 | Rows to enhance: 32777


Stage3: build 3A/3B/3C: 100%|██████████| 32777/32777 [5:20:04<00:00,  1.71it/s]   


[done] 3A run metrics: C:\Android Mobile App\ICST2026_Ext\run_metrics_v16_stage3_enhanced.csv
[done] 3B step breakdown: C:\Android Mobile App\ICST2026_Ext\run_steps_v16_stage3_breakdown.csv
[done] 3C run×style: C:\Android Mobile App\ICST2026_Ext\run_per_style_v1_stage3.csv


## Stage 4 — Build workload/test signature layer (artifact-first) and parse result/report artifacts

In [ ]:
"""
Stage 4 (ALIGNED with Stage 2/3 redesign)

What this version does (vs your prior Stage 4):
1) Step-category alignment with Stage 3:
   - Uses Stage 3 'category' values and includes gmd_setup/gradle where appropriate.
2) Provider/driver inference prefers run evidence over declared styles:
   - provider search order: steps_blob -> yaml -> styles
3) Adds provenance columns:
   - signature_inputs (artifacts|steps|yaml), has_steps_rows, has_yaml
4) Makes workload signature hash workload-centric (default excludes head_sha),
   and also outputs a commit-specific variant signature_hash_with_sha.

Inputs (ROOT_DIR):
  - run_metrics_v16_stage3_enhanced.csv
  - run_steps_v16_stage3_breakdown.csv

Output (ROOT_DIR):
  - run_workload_signature_v1.csv
"""

import base64
import csv
import hashlib
import random
import re
import time
import zipfile
from dataclasses import dataclass
from datetime import datetime, timezone
from io import BytesIO
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple, Union

import requests
import xml.etree.ElementTree as ET

try:
    from tqdm import tqdm
except ImportError:
    tqdm = None

# =========================
# CONFIG
# =========================
TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

ROOT_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext")

IN_STAGE3_CSV = ROOT_DIR / "run_metrics_v16_stage3_enhanced.csv"
IN_STEPS_CSV = ROOT_DIR / "run_steps_v16_stage3_breakdown.csv"
OUT_SIG_CSV = ROOT_DIR / "run_workload_signature_v1.csv"

MAX_TOKENS_TO_USE = 7

CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 90
MAX_RETRIES_PER_REQUEST = 8
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60
MAX_PAGES_PER_LIST = 2000

DOWNLOAD_AND_PARSE_ARTIFACTS = True
MAX_ARTIFACT_ZIP_BYTES = 25 * 1024 * 1024  # 25MB cap

FETCH_WORKFLOW_YAML = True
WORKFLOW_YAML_CACHE_MAX = 5000

# =========================
# Helpers
# =========================
BOM = "\ufeff"


def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def _clean_key(k: str) -> str:
    return (k or "").replace(BOM, "").strip()


def read_csv_rows(path: Path) -> Tuple[List[Dict[str, str]], List[str]]:
    if not path.exists():
        raise FileNotFoundError(f"CSV not found: {path}")
    with path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        rdr = csv.DictReader(f)
        raw_fields = rdr.fieldnames or []
        fields = [_clean_key(x) for x in raw_fields]
        rows: List[Dict[str, str]] = []
        for r in rdr:
            clean_row = {}
            for k, v in r.items():
                clean_row[_clean_key(k)] = (v or "")
            rows.append(clean_row)
    return rows, fields


def write_csv(path: Path, fieldnames: List[str], rows: List[Dict[str, str]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow(r)


def safe_lower(s: str) -> str:
    return (s or "").strip().lower()


def uniq_sorted(items: Iterable[str]) -> List[str]:
    s = {x.strip() for x in items if x and x.strip()}
    return sorted(s)


def stable_hash(parts: List[str]) -> str:
    blob = "\n".join([p.strip() for p in parts if p and p.strip()])
    return hashlib.sha1(blob.encode("utf-8", errors="ignore")).hexdigest()


def load_tokens_from_env_file(env_path: Path, max_tokens: int = 3) -> List[str]:
    if not env_path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {env_path}")
    tokens: List[str] = []
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN_") and v:
            tokens.append(v)
            if len(tokens) >= max_tokens:
                break
    if not tokens:
        raise ValueError(f"No tokens found in {env_path}. Expected keys like GITHUB_TOKEN_1=...")
    return tokens


# =========================
# GitHub API client (token rotation + retries)
# =========================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None


class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "run-metrics-stage4-signature/1.3",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        now = int(time.time())
        candidates = []
        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                candidates.append((0, i))
            else:
                if st.reset_epoch is not None and st.reset_epoch <= now:
                    candidates.append((0, i))
                else:
                    candidates.append((1, i))
        candidates.sort()
        return candidates[0][1]

    def _sleep_until_reset(self) -> None:
        now = int(time.time())
        resets = [st.reset_epoch for st in self.tokens if st.reset_epoch]
        if not resets:
            time.sleep(5)
            return
        soonest = min(resets)
        sleep_s = max(1, soonest - now + 2)
        print(f"[rate-limit] sleeping {sleep_s}s until reset...")
        time.sleep(sleep_s)

    def _backoff(self, attempt: int) -> None:
        sleep_s = min(BACKOFF_CAP_S, (BACKOFF_BASE_S ** attempt)) + random.random()
        time.sleep(sleep_s)

    def request(self, method: str, url: str, params: Optional[Dict] = None, stream: bool = False) -> Optional[requests.Response]:
        last_status = None
        for attempt in range(1, MAX_RETRIES_PER_REQUEST + 1):
            idx = self._pick_idx()
            st = self.tokens[idx]
            self.session.headers["Authorization"] = f"Bearer {st.token}"

            try:
                resp = self.session.request(
                    method,
                    url,
                    params=params,
                    timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S),
                    stream=stream
                )
            except requests.exceptions.RequestException:
                self._backoff(attempt)
                continue

            last_status = resp.status_code

            rem = resp.headers.get("X-RateLimit-Remaining")
            if rem is not None:
                try:
                    st.remaining = int(rem)
                except Exception:
                    pass
            rst = resp.headers.get("X-RateLimit-Reset")
            if rst is not None:
                try:
                    st.reset_epoch = int(rst)
                except Exception:
                    pass

            if resp.status_code == 404:
                return resp

            retry_after = resp.headers.get("Retry-After")
            if resp.status_code in (403, 429) and retry_after:
                try:
                    ra = int(retry_after)
                    time.sleep(min(BACKOFF_CAP_S, max(1, ra)) + random.random())
                    continue
                except Exception:
                    pass

            text_l = (resp.text or "").lower()
            if resp.status_code in (403, 429) and (
                "rate limit" in text_l
                or "secondary rate limit" in text_l
                or "abuse detection" in text_l
                or "too many requests" in text_l
            ):
                if any((t.remaining is None) or (t.remaining > 0) for t in self.tokens):
                    self._backoff(attempt)
                    continue
                self._sleep_until_reset()
                continue

            if resp.status_code in (500, 502, 503, 504):
                self._backoff(attempt)
                continue

            return resp

        print(f"[giveup] {method} {url} after {MAX_RETRIES_PER_REQUEST} tries (last_status={last_status})")
        return None

    def request_json(self, method: str, url: str, params: Optional[Dict] = None) -> Optional[Union[Dict, List]]:
        resp = self.request(method, url, params=params, stream=False)
        if resp is None:
            return None
        if resp.status_code == 404:
            return None
        if resp.status_code >= 400:
            return None
        try:
            return resp.json()
        except Exception:
            return None

    def paginate(self, url: str, params: Optional[Dict], item_key: str) -> Iterable[Dict]:
        page = 1
        while page <= MAX_PAGES_PER_LIST:
            p = dict(params or {})
            p.update({"per_page": 100, "page": page})
            data = self.request_json("GET", url, params=p)
            if data is None:
                return
            items = data if isinstance(data, list) else data.get(item_key, [])
            if not items:
                return
            for it in items:
                yield it
            if isinstance(items, list) and len(items) < 100:
                return
            page += 1


# =========================
# GitHub endpoints
# =========================
def list_run_artifacts(gh: GitHubClient, full_name: str, run_id: int) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/runs/{run_id}/artifacts"
    return list(gh.paginate(url, params={}, item_key="artifacts"))


def download_artifact_zip(gh: GitHubClient, full_name: str, artifact_id: int) -> Optional[bytes]:
    url = f"https://api.github.com/repos/{full_name}/actions/artifacts/{artifact_id}/zip"
    resp = gh.request("GET", url, params=None, stream=True)
    if resp is None or resp.status_code != 200:
        return None

    data = bytearray()
    try:
        for chunk in resp.iter_content(chunk_size=1024 * 128):
            if not chunk:
                continue
            data.extend(chunk)
            if len(data) > MAX_ARTIFACT_ZIP_BYTES:
                return None
    except Exception:
        return None

    return bytes(data)


def fetch_workflow_yaml(gh: GitHubClient, full_name: str, workflow_path: str, ref: str) -> str:
    url = f"https://api.github.com/repos/{full_name}/contents/{workflow_path.lstrip('/')}"
    data = gh.request_json("GET", url, params={"ref": ref})
    if not data or not isinstance(data, dict):
        return ""
    if data.get("encoding") == "base64" and data.get("content"):
        try:
            return base64.b64decode(data["content"]).decode("utf-8", errors="ignore")
        except Exception:
            return ""
    dl = data.get("download_url")
    if dl:
        resp = gh.request("GET", dl, params=None, stream=False)
        if resp and resp.status_code == 200:
            return resp.text or ""
    return ""


# =========================
# Signature inference patterns
# =========================
GRADLE_TASK_RE = re.compile(r"(?:\./gradlew|\bgradle[w]?\b)\s+([^\n\r#]+)", re.IGNORECASE)
GRADLE_TASK_TOKEN_RE = re.compile(r"(?:(?::[\w\-.]+)+|[\w\-.]+)", re.IGNORECASE)

THIRD_PARTY_PROVIDER_RE = re.compile(
    r"(firebase\s+test\s+lab|gcloud\s+firebase\s+test|flank|appcenter|browserstack|bstack|saucectl|maestro\s+cloud|emulator\.wtf)",
    re.IGNORECASE
)


def extract_gradle_tasks_from_yaml(yaml_text: str) -> List[str]:
    tasks: List[str] = []
    if not yaml_text:
        return tasks
    for m in GRADLE_TASK_RE.finditer(yaml_text):
        tail = (m.group(1) or "").strip()
        tail = tail.split("&&")[0].split(";")[0].strip()
        toks = tail.split()
        for t in toks:
            if t.startswith("-"):
                continue
            if t.lower() in ("cd", "echo", "export", "set"):
                continue
            if GRADLE_TASK_TOKEN_RE.fullmatch(t):
                tasks.append(t)

    keep = []
    for t in tasks:
        tl = t.lower()
        if any(k in tl for k in ("connected", "androidtest", "device", "check", "test", "manageddevice", "gmd")):
            keep.append(t)
    return uniq_sorted(keep)


def _infer_provider(styles: str, steps_blob: str, yaml_text: str) -> str:
    """
    Prefer run evidence over declared capability:
      1) steps_blob
      2) yaml_text
      3) styles
    """
    mprov = (
        THIRD_PARTY_PROVIDER_RE.search(steps_blob or "")
        or THIRD_PARTY_PROVIDER_RE.search(yaml_text or "")
        or THIRD_PARTY_PROVIDER_RE.search(styles or "")
    )
    return safe_lower(mprov.group(1)) if mprov else ""


def infer_test_driver(styles: str, steps_blob: str, yaml_text: str) -> str:
    """
    Robust driver inference:
      - If provider signals exist anywhere => third-party driver
      - Else if gradle-ish signals exist => gradle
      - Else unknown
    """
    s = safe_lower(styles)
    b = safe_lower(steps_blob)
    y = safe_lower(yaml_text)

    provider = _infer_provider(styles, steps_blob, yaml_text)
    if provider:
        if "firebase" in provider or "test lab" in provider or "gcloud" in provider:
            return "firebase_test_lab"
        if "flank" in provider:
            return "flank"
        if "appcenter" in provider:
            return "appcenter"
        if "browserstack" in provider or "bstack" in provider:
            return "browserstack"
        if "saucectl" in provider:
            return "sauce"
        if "maestro" in provider:
            return "maestro_cloud"
        if "emulator.wtf" in provider:
            return "emulator_wtf"
        return "third_party_unknown"

    # If style explicitly says third-party but provider not detected
    if "third-party" in s:
        return "third_party_unknown"

    # gradle-ish signals from steps or yaml
    if ("gradlew" in y) or ("./gradlew" in y) or ("gradle " in y) or ("connected" in b) or ("androidtest" in b) or ("manageddevice" in b) or ("gmd" in b):
        return "gradle"

    return "unknown"


def junit_metrics_from_zip_bytes(zip_bytes: bytes, max_testcases_for_fingerprint: int = 5000) -> Tuple[int, int, int, int, int, str]:
    suite_count = 0
    case_count = 0
    failed = 0
    errors = 0
    skipped = 0
    ids: List[str] = []

    try:
        with zipfile.ZipFile(BytesIO(zip_bytes)) as z:
            for name in z.namelist():
                nl = name.lower()
                if not nl.endswith(".xml"):
                    continue
                if ("junit" not in nl) and ("test-" not in nl) and ("test_results" not in nl) and ("androidtest" not in nl) and ("connected" not in nl) and ("surefire" not in nl):
                    continue
                try:
                    data = z.read(name)
                    root = ET.fromstring(data)
                except Exception:
                    continue

                if root.tag.lower().endswith("testsuite"):
                    suite_count += 1
                elif root.tag.lower().endswith("testsuites"):
                    suite_count += len(root.findall(".//testsuite"))

                suite_nodes = []
                if root.tag.lower().endswith("testsuite"):
                    suite_nodes = [root]
                elif root.tag.lower().endswith("testsuites"):
                    suite_nodes = root.findall(".//testsuite")

                for ts in suite_nodes:
                    try:
                        case_count += int(ts.get("tests") or 0)
                    except Exception:
                        pass
                    try:
                        failed += int(ts.get("failures") or 0)
                    except Exception:
                        pass
                    try:
                        errors += int(ts.get("errors") or 0)
                    except Exception:
                        pass
                    try:
                        skipped += int(ts.get("skipped") or 0)
                    except Exception:
                        pass

                if len(ids) < max_testcases_for_fingerprint:
                    for tc in root.findall(".//testcase"):
                        cls = tc.get("classname") or ""
                        nm = tc.get("name") or ""
                        if cls or nm:
                            ids.append(f"{cls}::{nm}")
                        if len(ids) >= max_testcases_for_fingerprint:
                            break

    except Exception:
        return (0, 0, 0, 0, 0, "")

    if case_count == 0:
        case_count = len(ids)

    fp = ""
    if ids:
        fp = hashlib.sha1("\n".join(sorted(set(ids))).encode("utf-8", errors="ignore")).hexdigest()

    return (suite_count, case_count, failed, errors, skipped, fp)


# =========================
# MAIN
# =========================
def main() -> None:
    if OUT_SIG_CSV.exists():
        OUT_SIG_CSV.unlink()

    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH, max_tokens=MAX_TOKENS_TO_USE)
    gh = GitHubClient(tokens)

    runs, _ = read_csv_rows(IN_STAGE3_CSV)
    steps, _ = read_csv_rows(IN_STEPS_CSV)

    if not runs:
        raise RuntimeError("Stage 3 enhanced CSV is empty.")
    if not steps:
        print("[warn] Steps breakdown CSV is empty; signatures will rely on artifacts + styles + YAML only.")

    # Map: (full_name, run_id) -> step rows
    steps_by_run: Dict[Tuple[str, str], List[Dict[str, str]]] = {}
    for s in steps:
        key = ((s.get("full_name") or "").strip(), (s.get("run_id") or "").strip())
        if not key[0] or not key[1]:
            continue
        steps_by_run.setdefault(key, []).append(s)

    yaml_cache: Dict[Tuple[str, str, str], str] = {}
    out_rows: List[Dict[str, str]] = []

    it = runs
    if tqdm is not None:
        it = tqdm(runs, desc="Stage4: workload signatures")

    # Align to Stage 3 categories:
    DRIVER_CATS = {"test", "third_party", "gmd_setup"}                # for provider/driver inference
    WORKLOAD_CATS = {"test", "third_party", "gmd_setup", "gradle"}    # for gradle task-ish hints

    for r in it:
        full_name = (r.get("full_name") or "").strip()
        run_id_s = (r.get("run_id") or "").strip()
        workflow_path = (r.get("workflow_path") or "").strip()
        head_sha = (r.get("head_sha") or "").strip()
        styles = (r.get("styles") or "")

        if not full_name or not run_id_s:
            continue

        run_key = (full_name, run_id_s)
        sr_list = steps_by_run.get(run_key, [])

        has_steps_rows = bool(sr_list)

        driver_step_names = [s.get("step_name", "") for s in sr_list if safe_lower(s.get("category", "")) in DRIVER_CATS]
        driver_job_names = [s.get("job_name", "") for s in sr_list if safe_lower(s.get("category", "")) in DRIVER_CATS]
        steps_blob_driver = "\n".join(driver_job_names + driver_step_names)

        workload_step_names = [s.get("step_name", "") for s in sr_list if safe_lower(s.get("category", "")) in WORKLOAD_CATS]
        workload_job_names = [s.get("job_name", "") for s in sr_list if safe_lower(s.get("category", "")) in WORKLOAD_CATS]
        steps_blob_workload = "\n".join(workload_job_names + workload_step_names)

        # --- artifacts ---
        artifacts = list_run_artifacts(gh, full_name, int(run_id_s)) or []
        artifact_names = uniq_sorted([a.get("name", "") for a in artifacts if a.get("name")])
        artifact_name_fingerprint = "|".join(artifact_names)

        junit_suites = 0
        junit_cases = 0
        junit_failed = 0
        junit_errors = 0
        junit_skipped = 0
        executed_tests_fingerprint = ""
        results_artifact_present = False
        results_artifact_types: Set[str] = set()

        for an in artifact_names:
            anl = (an or "").lower()
            if any(k in anl for k in ["test-result", "test_results", "junit", "androidtest", "instrumentation", "connected", "report", "reports", "results"]):
                results_artifact_present = True
                if "junit" in anl or "test-" in anl:
                    results_artifact_types.add("junit_xml_or_bundle")
                elif "report" in anl or "reports" in anl:
                    results_artifact_types.add("report_bundle")
                else:
                    results_artifact_types.add("results_bundle")

        if DOWNLOAD_AND_PARSE_ARTIFACTS and artifacts:
            cand = []
            for a in artifacts:
                n = (a.get("name") or "").lower()
                score = 0
                for kw in ["test-results", "test_results", "junit", "androidtest", "instrumentation", "connected", "report", "reports", "results"]:
                    if kw in n:
                        score += 1
                cand.append((score, a))
            cand.sort(key=lambda t: t[0], reverse=True)

            for score, a in cand[:5]:
                if score == 0:
                    continue
                try:
                    aid = int(a.get("id"))
                except Exception:
                    continue

                zip_bytes = download_artifact_zip(gh, full_name, aid)
                if not zip_bytes:
                    continue

                sc, cc, ff, ee, ss, fp = junit_metrics_from_zip_bytes(zip_bytes)
                junit_suites += sc
                junit_cases += cc
                junit_failed += ff
                junit_errors += ee
                junit_skipped += ss
                if fp and not executed_tests_fingerprint:
                    executed_tests_fingerprint = fp

                results_artifact_present = True
                results_artifact_types.add("parsed_zip")

        if executed_tests_fingerprint or (junit_cases > 0) or (junit_suites > 0):
            workload_evidence_level = "A"
        elif artifact_names:
            workload_evidence_level = "B"
        else:
            workload_evidence_level = "C"

        results_artifact_types_str = "|".join(sorted(results_artifact_types)) if results_artifact_types else ""

        # --- yaml + gradle tasks ---
        yaml_text = ""
        gradle_tasks: List[str] = []
        has_yaml = False

        if FETCH_WORKFLOW_YAML and workflow_path and head_sha:
            ck = (full_name, workflow_path, head_sha)
            if ck in yaml_cache:
                yaml_text = yaml_cache[ck]
            else:
                yaml_text = fetch_workflow_yaml(gh, full_name, workflow_path, ref=head_sha)
                if len(yaml_cache) < WORKFLOW_YAML_CACHE_MAX:
                    yaml_cache[ck] = yaml_text

            has_yaml = bool((yaml_text or "").strip())
            gradle_tasks = extract_gradle_tasks_from_yaml(yaml_text)

        provider = _infer_provider(styles, steps_blob_driver, yaml_text)
        test_driver = infer_test_driver(styles, steps_blob_driver, yaml_text)

        # signature kind:
        if artifact_names:
            signature_kind = "artifact"
        elif gradle_tasks or THIRD_PARTY_PROVIDER_RE.search(steps_blob_workload or "") or THIRD_PARTY_PROVIDER_RE.search(yaml_text or ""):
            signature_kind = "yaml"
        elif workload_step_names:
            signature_kind = "steps"
        else:
            signature_kind = "unknown"

        # module fingerprint from gradle tasks:
        modules: Set[str] = set()
        for t in gradle_tasks:
            if t.startswith(":"):
                parts = t.split(":")
                if len(parts) >= 2 and parts[1]:
                    modules.add(":" + parts[1])
        modules_sorted = sorted(modules)

        gradle_task_fingerprint = "|".join(gradle_tasks)
        module_fingerprint = "|".join(modules_sorted)

        # provenance:
        signature_inputs: List[str] = []
        if artifacts:
            signature_inputs.append("artifacts")
        if has_steps_rows:
            signature_inputs.append("steps")
        if has_yaml:
            signature_inputs.append("yaml")
        signature_inputs_str = "|".join(signature_inputs) if signature_inputs else ""

        # IMPORTANT: workload-centric hash (exclude head_sha)
        sig_parts = [
            f"kind={signature_kind}",
            f"driver={test_driver}",
            f"provider={provider}",
            f"artifact_names={artifact_name_fingerprint}",
            f"gradle_tasks={gradle_task_fingerprint}",
            f"modules={module_fingerprint}",
        ]
        signature_hash = stable_hash(sig_parts)
        signature_hash_with_sha = stable_hash(sig_parts + [f"head_sha={head_sha}"])

        out_rows.append({
            "full_name": full_name,
            "run_id": run_id_s,
            "workflow_path": workflow_path,
            "head_sha": head_sha,
            "styles": styles,

            "signature_kind": signature_kind,
            "signature_inputs": signature_inputs_str,
            "has_steps_rows": str(bool(has_steps_rows)),
            "has_yaml": str(bool(has_yaml)),

            "test_driver": test_driver,
            "provider": provider,

            "artifact_count": str(len(artifacts)),
            "artifact_name_fingerprint": artifact_name_fingerprint,

            "gradle_task_fingerprint": gradle_task_fingerprint,
            "module_fingerprint": module_fingerprint,

            "junit_suite_count": str(junit_suites),
            "junit_testcase_count": str(junit_cases),
            "junit_failed_count": str(junit_failed),
            "junit_error_count": str(junit_errors),
            "junit_skipped_count": str(junit_skipped),
            "executed_tests_fingerprint": executed_tests_fingerprint,

            "results_artifact_present": str(bool(results_artifact_present)),
            "results_artifact_types": results_artifact_types_str,
            "workload_evidence_level": workload_evidence_level,

            "signature_hash": signature_hash,
            "signature_hash_with_sha": signature_hash_with_sha,

            "stage4_extracted_at_utc": now_utc_iso(),
        })

    out_fields = [
        "full_name",
        "run_id",
        "workflow_path",
        "head_sha",
        "styles",

        "signature_kind",
        "signature_inputs",
        "has_steps_rows",
        "has_yaml",

        "test_driver",
        "provider",

        "artifact_count",
        "artifact_name_fingerprint",

        "gradle_task_fingerprint",
        "module_fingerprint",

        "junit_suite_count",
        "junit_testcase_count",
        "junit_failed_count",
        "junit_error_count",
        "junit_skipped_count",
        "executed_tests_fingerprint",

        "results_artifact_present",
        "results_artifact_types",
        "workload_evidence_level",

        "signature_hash",
        "signature_hash_with_sha",

        "stage4_extracted_at_utc",
    ]

    write_csv(OUT_SIG_CSV, out_fields, out_rows)
    print("Done.")
    print("Stage 4 output:", OUT_SIG_CSV)


if __name__ == "__main__":
    main()


Stage4: workload signatures: 100%|██████████| 32777/32777 [4:36:29<00:00,  1.98it/s]   


Done.
Stage 4 output: C:\Android Mobile App\ICST2026_Ext\run_workload_signature_v1.csv
